# Phân tích bình luận YouTube về điện thoại — bản code tổng hợp

**Đầu vào:** `src/ingest/data/raw/comments.jsonl` (schema v2, 15 trường). Notebook đọc một snapshot tại thời điểm chạy; không sửa, làm sạch, ghi đè hoặc cập nhật file raw.

**Cách chạy EDA hiện tại:** mở notebook trong repository, chạy các cell từ đầu đến hết mục **3.8 Threads and authors** (không chạy mục 4–6). Không dùng *Restart & Run All* nếu chưa muốn làm sạch, gán nhãn luật hoặc replay. Nếu kernel dùng working directory khác, hãy sửa `DATA_PATH` trong cell cấu hình.

Notebook này gộp các phiên bản code của nhóm vào một luồng duy nhất, mỗi bước dùng chung một `df`:

| Phần | Nội dung | Lấy từ |
|---|---|---|
| 1 | Cấu hình, nạp dữ liệu, cột dẫn xuất (giờ GMT+7, hãng) | `ds_inspect_comments_v2` |
| 2 | Kiểm tra chất lượng dữ liệu, scorecard | `ds_inspect_comments_v2` |
| 3 | EDA: thành phần, tương tác, thời gian, văn bản, từ vựng, tác giả | `ds_inspect_comments_v2` (thay cho `eda_smartphone_comments` và `EDA_Smartphone_Comments_Visual_Pro`) |
| 4–6 | Làm sạch, khía cạnh/cảm xúc và replay chỉ là phần tham khảo tiếp theo; chưa dùng để tạo dataset phân tích chính | — |
| 7 | Tóm tắt sinh từ dữ liệu | `ds_inspect_comments_v2` |

**Không có trong notebook này (có chủ đích):** F1, precision, recall, độ trễ, throughput. Các số đó trong mục 10, 11 của `eda_smartphone_comments` là số giả lập. Chúng chỉ được tính khi có tập vàng và log hệ thống.

**Phần 5, 6 là luật tham khảo, chưa phải mô hình đã kiểm định.** Crawler append comment mới vào JSONL theo từng trang API thành công. Với `run`, crawl bắt đầu sau khoảng 3 phút và lặp theo `crawl_every_min: 30`; hãy chạy lại notebook sau mỗi lần ingest hoặc theo lịch batch mong muốn để lấy snapshot mới.

**Không có trong EDA:** sentiment, nhãn khía cạnh đã kiểm định, dedup/cleaning làm thay đổi dữ liệu, F1/precision/recall, latency hoặc throughput.

## 1. Setup, loading, derived columns

In [ ]:
import json
import re
import unicodedata
import warnings
from collections import Counter
from itertools import chain
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

# ------------------------------------------------------------------ configuration
DATA_CANDIDATES = [
    Path("../ingest/data/raw/comments.jsonl"),
    Path("src/ingest/data/raw/comments.jsonl"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), DATA_CANDIDATES[0])
LOCAL_TZ = "Asia/Ho_Chi_Minh"   # published_at is UTC; the audience is (mostly) in Vietnam
MIN_N_MODEL = 300               # models with fewer comments are left out of comparison charts
TOP_K = 20                      # rows shown in "top-k" charts
SEED = 42

EXPECTED_COLUMNS = [
    "comment_id", "video_id", "parent_id", "parent_text", "model", "source", "text", "text_norm",
    "author_hash", "published_at", "like_count", "reply_count", "fetched_at", "ingested_at", "schema_version",
]

# model label -> brand. Rules are tried in order on the lower-cased label; labels such as
# "ip8|note8" are split on "|" and get brand "Mixed" if the parts disagree.
BRAND_RULES = [
    ("Apple",   r"^ip(?:\d|x|hone|se)"),
    ("Samsung", r"^(?:s\d+|note|galaxy|a\d{2}|zfold|zflip)"),
    ("Xiaomi",  r"^(?:mi\d|redmi|xiaomi|poco)"),
    ("Huawei",  r"^(?:p\d{2}|mate|huawei)"),
    ("Oppo",    r"^(?:reno|find|oppo)"),
    ("vivo",    r"^(?:vivo|x\d{2})"),
]
BRAND_COLORS = {
    "Apple": "#5F6B7A", "Samsung": "#1428A0", "Xiaomi": "#FF6900", "Huawei": "#C8102E",
    "Oppo": "#1D9A6C", "vivo": "#6C5CE7", "Mixed": "#9E9E9E", "Other": "#C9C9C9",
}

# ------------------------------------------------------------------ plotting / display defaults
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "axes.titleweight": "bold", "axes.titlesize": 12,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
warnings.filterwarnings("ignore", message="This pattern is interpreted as a regular expression")
rng = np.random.default_rng(SEED)

# ------------------------------------------------------------------ helpers
EMOJI_PAT = "[\U0001F000-\U0001FAFF\u2600-\u27BF\u2B50\u2B55\uFE0F\u200D]+"
EMOJI_RE = re.compile(EMOJI_PAT)
VI_CHARS = ("àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡ"
            "ùúụủũưừứựửữỳýỵỷỹđ")


def plot_label(s, maxlen=38):
    """Make arbitrary comment text safe for axis labels (emoji have no glyph in most fonts)."""
    s = EMOJI_RE.sub(lambda m: f"[emoji×{len(m.group())}]", str(s).replace("\n", " ").strip())
    s = s if s else "[blank]"
    return s if len(s) <= maxlen else s[: maxlen - 1] + "…"


def strip_accents(s: pd.Series) -> pd.Series:
    """Vectorised accent stripping (NFD, drop combining marks, đ→d). Input should be lower-cased."""
    return (s.str.normalize("NFD")
             .str.replace(r"[\u0300-\u036f]", "", regex=True)
             .str.replace("đ", "d", regex=False)
             .str.replace("Đ", "D", regex=False))


def strip_accents_str(s: str) -> str:
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    return s.replace("đ", "d").replace("Đ", "D")


def brand_of(label) -> str:
    brands = set()
    for part in str(label).split("|"):
        part = part.strip().lower()
        brands.add(next((b for b, pat in BRAND_RULES if re.match(pat, part)), "Other"))
    return brands.pop() if len(brands) == 1 else "Mixed"


def pct(x, d=1):
    return f"{x * 100:.{d}f}%"


def annotate_barh(ax, fmt="{:,.0f}", pad=0.01, fontsize=8):
    lo, hi = ax.get_xlim()
    span = hi - lo
    for p in ax.patches:
        w = p.get_width()
        if np.isfinite(w) and w != 0:
            ax.text(w + np.sign(w) * span * pad, p.get_y() + p.get_height() / 2, fmt.format(w),
                    va="center", ha="left" if w > 0 else "right", fontsize=fontsize)


def gini(values) -> float:
    v = np.sort(np.asarray(values, dtype=float))
    if v.sum() == 0:
        return 0.0
    n = len(v)
    return float((2 * np.arange(1, n + 1) - n - 1).dot(v) / (n * v.sum()))


def ccdf(s: pd.Series):
    """P(X >= x) for each distinct x."""
    v = np.sort(s.dropna().to_numpy())
    uniq, first_idx = np.unique(v, return_index=True)
    return uniq, (len(v) - first_idx) / len(v)


QA_LOG = []


def log_check(name, ok, detail="", severity="WARN"):
    """Record + print a data-quality check. severity: INFO < WARN < FAIL (used when the check does not pass)."""
    status = "PASS" if ok else severity
    QA_LOG.append({"check": name, "status": status, "detail": detail})
    print(f"[{status:<4}] {name}" + (f"  —  {detail}" if detail else ""))


LIKE_BINS, LIKE_LABELS = [-1, 0, 1, 2, 5, 10, 50, np.inf], ["0", "1", "2", "3–5", "6–10", "11–50", "51+"]
REPLY_BINS, REPLY_LABELS = [-1, 0, 1, 2, 5, 10, np.inf], ["0", "1", "2", "3–5", "6–10", "11+"]

### 1.1 Load the JSONL file
The loader reads line by line so that malformed lines are **reported instead of silently breaking the load**, and a wrong path raises a clear error (the original `try/except: print(e)` left `df` undefined and caused confusing errors later).

In [ ]:
def load_jsonl(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"{path.resolve()} not found — set DATA_PATH in the setup cell.")
    records, bad = [], []
    with path.open(encoding="utf-8") as fh:
        for i, line in enumerate(fh, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as exc:
                bad.append((i, str(exc)[:80]))
                continue
            if isinstance(obj, dict):
                records.append(obj)
            else:
                bad.append((i, "line is not a JSON object"))
    return pd.DataFrame.from_records(records), bad


raw, bad_lines = load_jsonl(DATA_PATH)
print(f"Loaded {len(raw):,} records from {DATA_PATH.name} ({DATA_PATH.stat().st_size / 1e6:.1f} MB)")
log_check("Every non-empty line is valid JSON", not bad_lines, f"{len(bad_lines)} bad line(s)", "FAIL")
if bad_lines:
    display(pd.DataFrame(bad_lines, columns=["line_no", "error"]).head(10))

missing_cols = [c for c in EXPECTED_COLUMNS if c not in raw.columns]
extra_cols = [c for c in raw.columns if c not in EXPECTED_COLUMNS]
log_check("All expected columns are present", not missing_cols, f"missing={missing_cols}", "FAIL")
log_check("No unexpected columns", not extra_cols, f"extra={extra_cols}", "INFO")

### 1.2 Type casting (explicit, with failure counts)

In [ ]:
df = raw.copy()
for c in missing_cols:                      # keep the notebook running; the check above already flagged it
    df[c] = np.nan

for c in ["published_at", "fetched_at", "ingested_at"]:
    parsed = pd.to_datetime(df[c], utc=True, errors="coerce")
    n_bad = int(parsed.isna().sum() - df[c].isna().sum())
    log_check(f"`{c}` parses as datetime", n_bad == 0, f"{n_bad:,} value(s) could not be parsed", "FAIL")
    df[c] = parsed

for c in ["like_count", "reply_count", "schema_version"]:
    num = pd.to_numeric(df[c], errors="coerce")
    n_bad = int(num.isna().sum() - df[c].isna().sum())
    log_check(f"`{c}` is numeric", n_bad == 0, f"{n_bad:,} non-numeric value(s)", "FAIL")
    df[c] = num

log_check("`like_count` has no negatives / nulls", bool((df["like_count"] >= 0).all()), "", "FAIL")

### 1.3 First look

In [ ]:
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns | memory ≈ {df.memory_usage(deep=True).sum() / 1e6:.1f} MB\n")
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "null_%": df.isna().mean() * 100,
    "n_unique": df.nunique(dropna=True),
})
display(overview)

const_cols = overview.index[overview["n_unique"] <= 1].tolist()
log_check("No constant columns", not const_cols, f"constant: {const_cols} → carry no information, drop for modelling", "INFO")

preview = df.head(5).copy()
for c in ["text", "parent_text", "text_norm"]:
    preview[c] = preview[c].fillna("").str.slice(0, 45)
for c in ["comment_id", "author_hash"]:
    preview[c] = preview[c].str.slice(0, 8) + "…"
display(preview)

### 1.4 Derived columns (non-destructive)
Everything below is **added**, raw columns are untouched.

In [ ]:
df["is_reply"] = df["parent_id"].notna()
df["comment_type"] = np.where(df["is_reply"], "reply", "top-level")

txt = df["text"].fillna("")
df["text_len"] = txt.str.len()
df["word_count"] = txt.str.split().str.len().fillna(0).astype(int)    # whitespace-separated chunks
df["text_ascii"] = strip_accents(txt.str.lower())                      # lower-case, no diacritics → robust matching

# local-time features (UTC → LOCAL_TZ)
local = df["published_at"].dt.tz_convert(LOCAL_TZ)
df["year"] = local.dt.year
df["month"] = local.dt.month
df["dow"] = local.dt.dayofweek            # 0 = Monday
df["hour"] = local.dt.hour
df["month_start"] = local.dt.tz_localize(None).dt.to_period("M").dt.to_timestamp()

# brand roll-up
brand_map = {m: brand_of(m) for m in df["model"].dropna().unique()}
df["brand"] = df["model"].map(brand_map).fillna("Other")

# engagement buckets
df["like_bucket"] = pd.cut(df["like_count"], LIKE_BINS, labels=LIKE_LABELS)
df["liked"] = df["like_count"] > 0

print("Comment types:", df["comment_type"].value_counts().to_dict())
print("Brand mapping (check that nothing is mis-assigned):")
brand_tbl = (pd.Series(brand_map).rename_axis("model").reset_index(name="brand")
             .groupby("brand")["model"].agg(lambda s: ", ".join(sorted(s))).to_frame("models"))
display(brand_tbl)

## 2. Data-quality checks
Each check prints `PASS / INFO / WARN / FAIL` and is collected in a scorecard at the end of this section (§2.9).

### 2.1 Missing and blank values
`NaN` is only one way to be missing — empty or whitespace-only strings are the other, and the original notebook did not look for them.

In [ ]:
str_cols = [c for c in EXPECTED_COLUMNS if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == object]
miss = pd.DataFrame({"null_n": df[EXPECTED_COLUMNS].isna().sum()})
miss["blank_n"] = pd.Series(
    {c: int((df[c].dropna().astype(str).str.strip() == "").sum()) for c in str_cols}
).reindex(miss.index).fillna(0).astype(int)
miss["null_%"] = miss["null_n"] / len(df) * 100
miss["blank_%"] = miss["blank_n"] / len(df) * 100
display(miss)

plot_df = miss[(miss["null_n"] + miss["blank_n"]) > 0].sort_values("null_%")
if len(plot_df):
    fig, ax = plt.subplots(figsize=(8, 0.55 * len(plot_df) + 1.6))
    ax.barh(plot_df.index, plot_df["null_%"], color="#4C72B0", label="null (NaN)")
    ax.barh(plot_df.index, plot_df["blank_%"], left=plot_df["null_%"], color="#DD8452", label="blank string ('' / whitespace)")
    ax.set_xlabel("% of rows")
    ax.set_title("Missing and blank values per column")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
else:
    print("No missing or blank values.")

# Is the missingness structural (explained by comment type) or random?
flags = pd.DataFrame({
    "parent_id null": df["parent_id"].isna(),
    "parent_text null": df["parent_text"].isna(),
    "reply_count null": df["reply_count"].isna(),
})
struct = flags.groupby(df["comment_type"]).sum()
struct.insert(0, "rows", df["comment_type"].value_counts())
print("\nMissingness by comment type (a structural pattern shows up as all-or-nothing rows):")
display(struct)

log_check("parent_id and parent_text are null together",
          bool((df["parent_id"].isna() == df["parent_text"].isna()).all()))
log_check("Every top-level comment has a reply_count",
          int(struct.loc["top-level", "reply_count null"]) == 0 if "top-level" in struct.index else True,
          f"{int(struct['reply_count null'].get('top-level', 0)):,} top-level rows without reply_count")
log_check("reply_count is null for replies only (expected: YouTube threads have 2 levels)",
          bool(df.loc[df["is_reply"], "reply_count"].isna().all()) if df["is_reply"].any() else True,
          "", "INFO")

> **Decision:** `reply_count` is **not** filled with 0. For a reply it means *not applicable*, not *no replies*.
> Filling it (as the original notebook did) pulls the mean/percentiles of `reply_count` toward 0 by ~43 % and makes every
> reply-count statistic uninterpretable. From here on, reply-count statistics use top-level comments only.

### 2.2 Keys and referential integrity
* `comment_id` should be unique.
* Every reply's `parent_id` should point to a top-level comment present in the file, and `parent_text` / `video_id` / `model` should agree with that parent.

In [ ]:
dup_id = df["comment_id"].duplicated(keep=False)
log_check("comment_id is unique", not dup_id.any(),
          f"{dup_id.sum():,} rows share {df.loc[dup_id, 'comment_id'].nunique():,} id(s)", "FAIL")

content_cols = [c for c in EXPECTED_COLUMNS if c not in ("fetched_at", "ingested_at")]
n_full_dup = int(df.duplicated(subset=content_cols).sum())
log_check("No fully duplicated records (ignoring fetch/ingest timestamps)", n_full_dup == 0, f"{n_full_dup:,} duplicate rows")

top_unique = df.loc[~df["is_reply"]].drop_duplicates("comment_id").set_index("comment_id")
rep = df.loc[df["is_reply"]]
resolvable = rep["parent_id"].isin(top_unique.index)
r = rep.loc[resolvable]

rates = {
    "parent found in file": resolvable.mean() if len(rep) else np.nan,
    "parent_text == parent's text": (r["parent_text"] == r["parent_id"].map(top_unique["text"])).mean() if len(r) else np.nan,
    "same video_id as parent": (r["video_id"] == r["parent_id"].map(top_unique["video_id"])).mean() if len(r) else np.nan,
    "same model as parent": (r["model"] == r["parent_id"].map(top_unique["model"])).mean() if len(r) else np.nan,
}
rates = pd.Series(rates) * 100
print("Share of replies passing each check (denominator: all replies for the first, resolvable replies for the rest):")
display(rates.to_frame("pass_%"))

fig, ax = plt.subplots(figsize=(8, 2.8))
colors = ["#55A868" if v >= 99 else "#DD8452" if v >= 90 else "#C44E52" for v in rates.fillna(0)]
ax.barh(rates.index[::-1], rates.fillna(0).values[::-1], color=colors[::-1])
ax.set_xlim(0, 100)
ax.set_xlabel("% of replies")
ax.set_title("Reply → parent referential integrity")
annotate_barh(ax, "{:.1f}%", pad=0.01)
plt.tight_layout()
plt.show()

log_check("All replies point to a top-level comment present in the file", bool(resolvable.all()),
          f"{(~resolvable).sum():,} orphan repl(ies) ({pct((~resolvable).mean() if len(rep) else 0)})")
log_check("parent_text matches the parent's text", bool(np.isnan(rates.iloc[1]) or rates.iloc[1] > 99), f"{rates.iloc[1]:.1f}% match")
log_check("A reply never points to itself", not bool((df["parent_id"] == df["comment_id"]).any()), "", "FAIL")

### 2.3 Is `reply_count` consistent with the replies actually in the file?
`reply_count` is YouTube's number at fetch time; the file may contain fewer replies (crawl truncation, deleted replies).
This tells us how much of each thread we can actually analyse.

In [ ]:
observed = rep.groupby("parent_id").size()
tl = df.loc[~df["is_reply"] & df["reply_count"].notna(), ["comment_id", "reply_count"]].copy()
tl["observed"] = tl["comment_id"].map(observed).fillna(0).astype(int)
tl["reported"] = tl["reply_count"].astype(int)

status = np.select(
    [tl["reported"].eq(0) & tl["observed"].eq(0),
     tl["observed"] > tl["reported"],
     tl["observed"].eq(tl["reported"]),
     tl["observed"].eq(0),
     tl["observed"] < tl["reported"]],
    ["no replies (reported = 0)", "more observed than reported", "all replies captured", "none captured", "partially captured"],
    default="other")
status_share = pd.Series(status).value_counts(normalize=True).mul(100)
display(status_share.to_frame("% of top-level comments"))

total_rep = int(tl["reported"].sum())
with_replies = tl[tl["reported"] > 0]
full_capture_share = float((with_replies["observed"] == with_replies["reported"]).mean()) if len(with_replies) else float("nan")
print(f"Replies reported by YouTube: {total_rep:,} | present in file: {len(rep):,} "
      f"→ overall capture ratio ≈ {len(rep) / max(total_rep, 1):.1%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1.1, 1]})
ax = axes[0]
hb = ax.hexbin(np.log1p(tl["reported"]), np.log1p(tl["observed"]), gridsize=35, bins="log", mincnt=1, cmap="viridis")
lim = np.log1p(max(tl["reported"].max(), tl["observed"].max(), 1))
ax.plot([0, lim], [0, lim], "r--", lw=1, label="observed = reported")
ticks = [0, 1, 2, 5, 10, 25, 50, 100]
ticks = [t for t in ticks if np.log1p(t) <= lim]
ax.set_xticks(np.log1p(ticks)); ax.set_xticklabels(ticks)
ax.set_yticks(np.log1p(ticks)); ax.set_yticklabels(ticks)
ax.set_xlabel("reply_count reported by YouTube"); ax.set_ylabel("replies present in file")
ax.set_title("Reported vs observed replies per top-level comment (hexbin, log colour)")
ax.legend(loc="upper left")
fig.colorbar(hb, ax=ax, label="comments (log)")

ax = axes[1]
order = ["no replies (reported = 0)", "all replies captured", "partially captured", "none captured", "more observed than reported"]
vals = status_share.reindex(order).fillna(0)
ax.barh(vals.index[::-1], vals.values[::-1], color="#4C72B0")
ax.set_xlabel("% of top-level comments")
ax.set_title("Thread completeness")
annotate_barh(ax, "{:.1f}%")
plt.tight_layout()
plt.show()

n_inconsistent = int((tl["observed"] > tl["reported"]).sum())
log_check("No top-level comment has more replies in the file than reply_count", n_inconsistent == 0,
          f"{n_inconsistent:,} top-level comment(s) ({pct(n_inconsistent / max(len(tl), 1), 3)}) — usually duplicated reply rows")

incomplete = float(((tl["observed"] < tl["reported"])).mean())
log_check("Threads are fully captured", incomplete < 0.02, f"{pct(incomplete)} of top-level comments have missing replies", "INFO")

### 2.4 Duplicate content
The original notebook counted *any* repeated text as a duplicate, which mostly flags harmless short comments ("hay", "ok", "."). Here duplicates are split into:
* **trivial** — normalised text ≤ 4 characters (`.`, `ok`, `hay`, an emoji…). Expected to repeat; not a data problem.
* **substantive** — longer texts repeated verbatim: copy-paste, spam, or ingestion duplicates. These matter for modelling (train/test leakage).

`text_key` normalises case, accents, punctuation and whitespace, so "Chuẩn!" and "chuan" collapse into one group.

In [ ]:
key = df["text_ascii"].str.replace(r"[^0-9a-z]+", " ", regex=True).str.strip()
df["text_key"] = key.where(key != "", txt.str.strip())      # emoji/punctuation-only comments keep their raw text
df["dup_group_n"] = df.groupby("text_key")["comment_id"].transform("size")
df["is_trivial"] = df["text_key"].str.len() <= 4
df["dup_ingest"] = df.duplicated(subset=["video_id", "author_hash", "text", "published_at"], keep="first")
df["dup_same_author"] = df.duplicated(subset=["video_id", "author_hash", "text"], keep="first")

dup_mask = df["dup_group_n"] > 1
cat = pd.DataFrame({
    "rows": [dup_mask.sum(), (dup_mask & df["is_trivial"]).sum(), (dup_mask & ~df["is_trivial"]).sum()],
    "distinct texts": [df.loc[dup_mask, "text_key"].nunique(),
                       df.loc[dup_mask & df["is_trivial"], "text_key"].nunique(),
                       df.loc[dup_mask & ~df["is_trivial"], "text_key"].nunique()],
}, index=["any repeated text", "  … trivial (≤ 4 chars)", "  … substantive"])
cat["% of all rows"] = cat["rows"] / len(df) * 100
display(cat)

print(f"Same video + author + text + timestamp (ingestion duplicates): {int(df['dup_ingest'].sum()):,}")
print(f"Same video + author + text (reposts / spam)                   : {int(df['dup_same_author'].sum()):,}")

grp = (df[dup_mask].groupby("text_key")
       .agg(n=("comment_id", "size"), authors=("author_hash", "nunique"), videos=("video_id", "nunique"),
            example=("text", "first"), trivial=("is_trivial", "first"))
       .sort_values("n", ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5), gridspec_kw={"width_ratios": [1, 1, 0.9]})
for ax, flag, title, color in [(axes[0], True, "Most repeated — trivial texts", "#8C8C8C"),
                               (axes[1], False, "Most repeated — substantive texts", "#C44E52")]:
    g = grp[grp["trivial"] == flag].head(12)[::-1]
    ax.barh([plot_label(t) for t in g["example"]], g["n"], color=color)
    ax.set_title(title); ax.set_xlabel("occurrences (log scale)")
    ax.set_xscale("log"); ax.set_xlim(left=1)
    for p in ax.patches:
        ax.text(p.get_width() * 1.05, p.get_y() + p.get_height() / 2, f"{p.get_width():,.0f}", va="center", fontsize=8)
    ax.margins(x=0.15)
ax = axes[2]
sizes = grp["n"].value_counts().sort_index()
ax.scatter(sizes.index, sizes.values, s=18, color="#4C72B0")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("size of duplicate group (log)"); ax.set_ylabel("number of groups (log)")
ax.set_title("Duplicate-group size distribution")
plt.tight_layout()
plt.show()

print("Substantive repeated texts (most frequent) — `authors` ≪ `n` suggests one account spamming:")
display(grp[~grp["trivial"]].head(10)[["n", "authors", "videos", "example"]]
        .assign(example=lambda d: d["example"].str.slice(0, 90)))

sub_share = float((dup_mask & ~df["is_trivial"]).mean())
log_check("Substantive duplicate comments are rare (< 1 % of rows)", sub_share < 0.01,
          f"{pct(sub_share, 2)} of rows; consider de-duplicating on `text_key` before train/test splits", "WARN")
log_check("No ingestion duplicates (same video+author+text+timestamp)", int(df["dup_ingest"].sum()) == 0,
          f"{int(df['dup_ingest'].sum()):,} rows")

### 2.5 Timestamp integrity

In [ ]:
pub, fet, ing = df["published_at"], df["fetched_at"], df["ingested_at"]
print(f"published_at range (UTC): {pub.min()}  →  {pub.max()}")
print(f"fetched_at   range (UTC): {fet.min()}  →  {fet.max()}   ({fet.nunique():,} distinct value(s))")
print(f"ingested_at  range (UTC): {ing.min()}  →  {ing.max()}   ({ing.nunique():,} distinct value(s))")

n_future = int((pub > fet).sum())
n_fetch_after_ingest = int((fet > ing).sum())
parent_pub = rep["parent_id"].map(top_unique["published_at"])
n_reply_before_parent = int((rep["published_at"] < parent_pub).sum())
n_nat = int(pub.isna().sum())

log_check("published_at is never null", n_nat == 0, f"{n_nat:,} null", "FAIL")
log_check("published_at ≤ fetched_at (no comments from the future)", n_future == 0, f"{n_future:,} violations", "FAIL")
log_check("fetched_at ≤ ingested_at", n_fetch_after_ingest == 0, f"{n_fetch_after_ingest:,} violations")
log_check("Replies are not older than their parent", n_reply_before_parent == 0, f"{n_reply_before_parent:,} violations")
log_check("fetched_at varies (a single snapshot means no time-of-crawl effects can be studied)",
          fet.nunique() > 1, f"{fet.nunique()} distinct fetch time(s)", "INFO")
if pub.max() < fet.max() - pd.Timedelta(days=60):
    print(f"Note: newest comment is {(fet.max() - pub.max()).days} days older than the fetch time — "
          "the latest period may be under-collected.")

### 2.6 Text hygiene
What is inside `text`: URLs, HTML remnants, YouTube timestamps, emoji, elongated words, ALL-CAPS, non-NFC Unicode (a classic problem for Vietnamese, where "ả" can be one code point or two), and **Vietnamese written without diacritics** (very common in comments and the main reason naive NLP fails).

In [ ]:
FLAGS = {
    "URL": r"https?://|www\.",
    "HTML entity / tag": r"&(?:amp|lt|gt|quot|apos|#\d+);|</?(?:br|a|b|i|span)\b",
    "@mention": r"(?<!\w)@\w+",
    "YouTube timestamp (m:ss)": r"(?<!\d)\d{1,2}:\d{2}(?!\d)",
    "#hashtag": r"(?<!\w)#\w+",
    "Elongated chars (aaaa)": r"(.)\1{3,}",
    "Contains emoji": EMOJI_PAT,
    "Contains digits": r"\d",
    "Contains '?'": r"\?",
    "Replacement char (mojibake)": "\ufffd",
}
flag_df = pd.DataFrame({name: txt.str.contains(pat, regex=True) for name, pat in FLAGS.items()})
n_letters = txt.str.count(r"[^\W\d_]")
flag_df["ALL CAPS (≥ 5 letters)"] = (txt.str.upper() == txt) & (n_letters >= 5)
flag_df["Not NFC-normalised"] = txt != txt.str.normalize("NFC")
flag_df["Blank / whitespace only"] = txt.str.strip() == ""
flag_df["Very short (≤ 2 chars)"] = txt.str.strip().str.len() <= 2
flag_df["Very long (> 1,000 chars)"] = txt.str.len() > 1000

df["has_emoji"] = flag_df["Contains emoji"]
df["has_url"] = flag_df["URL"]

has_vi = txt.str.lower().str.contains("[" + VI_CHARS + "]", regex=True)
df["script_style"] = np.select(
    [n_letters == 0, has_vi],
    ["No letters (emoji / punctuation / digits)", "Vietnamese with diacritics"],
    default="Latin letters, no diacritics")

flag_share = flag_df.mean().mul(100).sort_values()
style_share = df["script_style"].value_counts(normalize=True).mul(100)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw={"width_ratios": [1.5, 1]})
ax = axes[0]
ax.barh(flag_share.index, flag_share.values, color="#4C72B0")
ax.set_xlabel("% of comments"); ax.set_title("Text-hygiene flags (a comment can have several)")
annotate_barh(ax, "{:.2f}%")
ax = axes[1]
STYLE_COLORS = {"Vietnamese with diacritics": "#55A868", "Latin letters, no diacritics": "#DD8452",
                "No letters (emoji / punctuation / digits)": "#C9C9C9"}
ax.barh(style_share.index[::-1], style_share.values[::-1], color=[STYLE_COLORS[i] for i in style_share.index[::-1]])
ax.set_xlabel("% of comments"); ax.set_title("Writing style")
annotate_barh(ax, "{:.1f}%")
plt.tight_layout()
plt.show()

log_check("No mojibake (U+FFFD) in text", flag_df["Replacement char (mojibake)"].sum() == 0,
          f"{int(flag_df['Replacement char (mojibake)'].sum()):,} rows", "FAIL")
log_check("All text is NFC-normalised", flag_df["Not NFC-normalised"].sum() == 0,
          f"{int(flag_df['Not NFC-normalised'].sum()):,} rows need `unicodedata.normalize('NFC', …)` before tokenising")
log_check("No HTML entities / tags left in text", flag_df["HTML entity / tag"].sum() == 0,
          f"{int(flag_df['HTML entity / tag'].sum()):,} rows — unescape with html.unescape()")
log_check("No blank comments", flag_df["Blank / whitespace only"].sum() == 0, f"{int(flag_df['Blank / whitespace only'].sum()):,} rows", "WARN")
log_check("Most comments keep Vietnamese diacritics", style_share.get("Vietnamese with diacritics", 0) > 50,
          f"{style_share.get('Vietnamese with diacritics', 0):.1f}% with diacritics; "
          f"{style_share.get('Latin letters, no diacritics', 0):.1f}% without → plan for diacritic-insensitive features", "INFO")

# --- the pre-computed `text_norm` column -------------------------------------------------
tn = df["text_norm"].fillna("")
tn_blank = (tn.str.strip() == "") & (txt.str.strip() != "")
df["text_norm_blank"] = tn_blank
log_check("`text_norm` is populated whenever `text` is non-empty", not tn_blank.any(),
          f"{pct(tn_blank.mean())} of rows have an empty text_norm — the normalisation step looks incomplete", "WARN")
if tn_blank.any():
    print("Examples (text present, text_norm empty):")
    display(df.loc[tn_blank, ["model", "text", "text_norm"]].head(6).assign(text=lambda d: d["text"].str.slice(0, 80)))
    by_year = df.groupby("year")["text_norm_blank"].mean().mul(100)
    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.bar(by_year.index.astype(int).astype(str), by_year.values, color="#C44E52")
    ax.set_ylabel("% rows with empty text_norm"); ax.set_xlabel("year (local time)")
    ax.set_title("Where is `text_norm` missing?")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

### 2.7 Label hygiene: `model`, `brand`, `video_id`, `source`
`model` is a free-text label. Composite labels (`ip8|note8`) mean the video discusses several phones, so a comment cannot be attributed to a single model.

In [ ]:
composite = df["model"].str.contains("|", regex=False)
label_type = pd.Series(np.select(
    [composite, df["brand"].eq("Other")], ["Composite label (a|b)", "Brand not recognised"], default="Single model, brand recognised"),
    index=df.index)
lt = label_type.value_counts()
display(lt.to_frame("comments").assign(**{"%": lambda d: d["comments"] / len(df) * 100}))

if composite.any():
    print("Composite labels:")
    display(df[composite].groupby("model").agg(comments=("comment_id", "size"), videos=("video_id", "nunique"), brand=("brand", "first")))
unrec = sorted(df.loc[df["brand"].eq("Other"), "model"].unique())
if unrec:
    print("Models not matched by BRAND_RULES — extend the rules in the setup cell:", unrec)

videos_per_model_label = df.groupby("video_id")["model"].nunique()
n_case_variants = df["model"].nunique() - df["model"].str.lower().nunique()
log_check("Each video maps to exactly one model label", bool((videos_per_model_label == 1).all()),
          f"{int((videos_per_model_label > 1).sum()):,} video(s) carry several labels")
log_check("No composite model labels", not composite.any(),
          f"{pct(composite.mean())} of comments have a composite label (e.g. {df.loc[composite, 'model'].iloc[0] if composite.any() else '-'}); "
          "decide: drop, split, or keep as a separate 'multi-model' class", "WARN")
log_check("All models map to a known brand", not unrec, f"unmapped: {unrec}", "WARN")
log_check("Model labels have consistent casing", n_case_variants == 0, f"{n_case_variants} label(s) differ only by case", "WARN")
log_check("`source` has a single value (YouTube only)", df["source"].nunique() == 1, f"values: {df['source'].unique().tolist()}", "INFO")

fig, ax = plt.subplots(figsize=(8, 2.4))
ax.barh(lt.index[::-1], lt.values[::-1], color="#4C72B0")
ax.set_xlabel("comments"); ax.set_title("What kind of model label do comments carry?")
annotate_barh(ax)
plt.tight_layout()
plt.show()

### 2.8 Authors (`author_hash`)
Format, activity concentration, and a simple spam signal: the share of an author's comments that repeat an earlier text.

In [ ]:
valid_hash = df["author_hash"].fillna("").str.fullmatch(r"[0-9a-f]{64}")
log_check("author_hash is a 64-char hex digest everywhere", bool(valid_hash.all()), f"{int((~valid_hash).sum()):,} malformed", "WARN")

per_author = (df.groupby("author_hash")
              .agg(n=("comment_id", "size"), models=("model", "nunique"), videos=("video_id", "nunique"),
                   uniq_texts=("text_key", "nunique"))
              .assign(repeat_ratio=lambda d: 1 - d["uniq_texts"] / d["n"]))
n_auth = len(per_author)
one_shot = (per_author["n"] == 1).mean()
top1_share = per_author["n"].sort_values(ascending=False).head(max(int(n_auth * 0.01), 1)).sum() / len(df)
print(f"{n_auth:,} distinct authors | {pct(one_shot)} wrote exactly one comment | "
      f"top 1 % of authors wrote {pct(top1_share)} of all comments | max = {per_author['n'].max():,} comments by one author")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
ax = axes[0]
xs, ys = ccdf(per_author["n"])
ax.loglog(xs, ys, marker=".", ms=4, lw=1, color="#4C72B0")
ax.set_xlabel("comments per author (log)"); ax.set_ylabel("P(authors with ≥ x comments) (log)")
ax.set_title("Author activity is heavy-tailed")
ax = axes[1]
active = per_author[per_author["n"] >= 5]
if len(active):
    ax.hist(active["repeat_ratio"], bins=20, color="#DD8452", edgecolor="white")
    ax.set_xlabel("share of the author's comments that repeat one of their earlier texts")
    ax.set_ylabel("authors"); ax.set_title(f"Repetition among authors with ≥ 5 comments (n = {len(active):,})")
else:
    ax.text(0.5, 0.5, "no author with ≥ 5 comments", ha="center"); ax.set_axis_off()
plt.tight_layout()
plt.show()

print("Most active authors:")
display(per_author.sort_values("n", ascending=False).head(8)
        .rename(index=lambda h: h[:8] + "…")[["n", "models", "videos", "repeat_ratio"]])
suspects = per_author[(per_author["n"] >= 10) & (per_author["repeat_ratio"] > 0.5)]
log_check("No obvious repeat-posting accounts (≥ 10 comments and > 50 % repeats)", suspects.empty,
          f"{len(suspects):,} account(s) → candidates for spam filtering", "WARN")

### 2.9 Data-quality scorecard

In [ ]:
scorecard = pd.DataFrame(QA_LOG)
order = {"FAIL": 0, "WARN": 1, "INFO": 2, "PASS": 3}
scorecard = scorecard.sort_values("status", key=lambda s: s.map(order), kind="stable").reset_index(drop=True)
display(scorecard)

counts = scorecard["status"].value_counts().reindex(["FAIL", "WARN", "INFO", "PASS"]).fillna(0)
fig, ax = plt.subplots(figsize=(7, 2.4))
ax.barh(counts.index[::-1], counts.values[::-1], color=["#55A868", "#8C8C8C", "#DD8452", "#C44E52"])
ax.set_xlabel("number of checks"); ax.set_title("Data-quality checks by outcome")
annotate_barh(ax)
plt.tight_layout()
plt.show()

## 3. Exploratory data analysis
### 3.1 Composition: comment type, brand, model, video

In [ ]:
top = df[~df["is_reply"]]
rep = df[df["is_reply"]]

type_ct = df["comment_type"].value_counts()
brand_ct = df["brand"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [0.8, 1.2]})
ax = axes[0]
ax.bar(type_ct.index, type_ct.values, color=["#4C72B0", "#DD8452"][: len(type_ct)])
for i, v in enumerate(type_ct.values):
    ax.text(i, v, f"{v:,}\n({v / len(df):.1%})", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, type_ct.max() * 1.18); ax.set_ylabel("comments"); ax.set_title("Top-level comments vs replies")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{int(v):,}"))
ax = axes[1]
ax.barh(brand_ct.index[::-1], brand_ct.values[::-1], color=[BRAND_COLORS[b] for b in brand_ct.index[::-1]])
ax.set_xlabel("comments"); ax.set_title("Comments by brand (rolled up from `model`)")
annotate_barh(ax)
plt.tight_layout()
plt.show()

print("source:", df["source"].value_counts().to_dict(), "→ single-source dataset, nothing to plot." if df["source"].nunique() == 1 else "")

In [ ]:
mc = (df.groupby(["model", "brand"]).agg(comments=("comment_id", "size"), videos=("video_id", "nunique"))
        .reset_index().sort_values("comments"))
fig, ax = plt.subplots(figsize=(10, 0.27 * len(mc) + 1.8))
ax.barh(mc["model"], mc["comments"], color=mc["brand"].map(BRAND_COLORS))
ax.axvline(MIN_N_MODEL, color="red", ls="--", lw=1)
ax.text(MIN_N_MODEL, len(mc) - 0.3, f"  MIN_N_MODEL = {MIN_N_MODEL}", color="red", fontsize=8, va="bottom")
ax.set_xlabel("comments"); ax.set_title("Comments per model (colour = brand)")
annotate_barh(ax)
handles = [plt.Rectangle((0, 0), 1, 1, color=BRAND_COLORS[b]) for b in brand_ct.index]
ax.legend(handles, brand_ct.index, loc="lower right", title="brand")
plt.tight_layout()
plt.show()

shares = mc.sort_values("comments", ascending=False)["comments"].cumsum() / len(df)
print(f"{len(mc)} models | top-5 models hold {pct(shares.iloc[min(4, len(shares) - 1)])} of comments | "
      f"{(mc['comments'] < MIN_N_MODEL).sum()} model(s) fall below MIN_N_MODEL and are excluded from comparison charts.")

In [ ]:
per_video = df.groupby("video_id").agg(comments=("comment_id", "size"), model=("model", "first"), brand=("brand", "first"))
print(f"{len(per_video):,} videos | comments per video: median {per_video['comments'].median():.0f}, "
      f"mean {per_video['comments'].mean():.1f}, max {per_video['comments'].max():,}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
ax = axes[0]
ax.hist(np.log10(per_video["comments"]), bins=30, color="#4C72B0", edgecolor="white")
ticks = [1, 3, 10, 30, 100, 300, 1000, 3000]
ticks = [t for t in ticks if np.log10(t) <= np.log10(per_video["comments"].max()) + 0.2]
ax.set_xticks(np.log10(ticks)); ax.set_xticklabels(ticks)
ax.set_xlabel("comments per video (log scale)"); ax.set_ylabel("videos"); ax.set_title("Comments per video")
ax = axes[1]
vpm = df.groupby("model")["video_id"].nunique().sort_values(ascending=False).head(TOP_K)[::-1]
ax.barh(vpm.index, vpm.values, color=[BRAND_COLORS[brand_map.get(m, "Other")] for m in vpm.index])
ax.set_xlabel("distinct videos"); ax.set_title(f"Videos per model (top {len(vpm)}) — coverage breadth")
annotate_barh(ax)
plt.tight_layout()
plt.show()

print("Videos with the most comments (a few videos can dominate a model's data):")
display(per_video.sort_values("comments", ascending=False).head(8))

### 3.2 Engagement distributions (likes, replies)
Both are zero-inflated and extremely heavy-tailed, so means and boxplots are poor summaries. We use **separate stats for top-level comments and replies**, percentile tables, bucketed bars, a log-log **CCDF** (shows the tail honestly), and a **Lorenz curve** (concentration of likes).

In [ ]:
def describe_engagement(s: pd.Series) -> pd.Series:
    s = s.dropna()
    out = {"n": len(s), "mean": s.mean(), "std": s.std(), "% zero": (s == 0).mean() * 100}
    for q in (0.5, 0.75, 0.9, 0.95, 0.99, 0.999):
        out[f"p{q * 100:g}"] = s.quantile(q)
    out.update({"max": s.max(), "sum": s.sum()})
    return pd.Series(out)


eng = pd.DataFrame({
    "likes — top-level": describe_engagement(top["like_count"]),
    "likes — replies": describe_engagement(rep["like_count"]),
    "replies received — top-level": describe_engagement(top["reply_count"]),
}).T
display(eng)

thr = top["like_count"].quantile(0.99)
print(f"Top-level comments above the 99th percentile of likes (> {thr:g}): {(top['like_count'] > thr).sum():,} "
      f"— with discrete data ties at the threshold make the exact count differ from 1 %.")

likes_sorted = np.sort(df["like_count"].to_numpy())
cum_likes = np.cumsum(likes_sorted) / likes_sorted.sum() if likes_sorted.sum() else np.zeros_like(likes_sorted)
n_all = len(likes_sorted)
share_top1 = likes_sorted[-max(int(n_all * 0.01), 1):].sum() / max(likes_sorted.sum(), 1)
share_top10 = likes_sorted[-max(int(n_all * 0.10), 1):].sum() / max(likes_sorted.sum(), 1)
G = gini(likes_sorted)
print(f"Gini of like_count = {G:.3f} | top 1 % of comments hold {pct(share_top1)} of all likes, top 10 % hold {pct(share_top10)}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# (a) like buckets by comment type
ax = axes[0, 0]
lb = (pd.crosstab(df["comment_type"], df["like_bucket"], normalize="index") * 100).stack().rename("pct").reset_index()
sns.barplot(data=lb, x="like_bucket", y="pct", hue="comment_type", hue_order=["top-level", "reply"], ax=ax,
            palette={"top-level": "#4C72B0", "reply": "#DD8452"})
ax.set_xlabel("likes received"); ax.set_ylabel("% of comments"); ax.set_title("Likes per comment (bucketed)")
ax.legend(title="")

# (b) reply buckets, top-level only
ax = axes[0, 1]
rb = pd.cut(top["reply_count"], REPLY_BINS, labels=REPLY_LABELS).value_counts(normalize=True).sort_index() * 100
ax.bar(rb.index.astype(str), rb.values, color="#4C72B0")
for i, v in enumerate(rb.values):
    ax.text(i, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=8)
ax.set_xlabel("replies received"); ax.set_ylabel("% of top-level comments"); ax.set_title("Replies per top-level comment (bucketed)")

# (c) CCDF
ax = axes[1, 0]
for name, s, c in [("likes — top-level", top["like_count"], "#4C72B0"),
                   ("likes — replies", rep["like_count"], "#DD8452"),
                   ("replies received — top-level", top["reply_count"], "#55A868")]:
    xs, ys = ccdf(s)
    m = xs >= 1
    ax.loglog(xs[m], ys[m], marker=".", ms=4, lw=1, label=name, color=c)
ax.set_xlabel("x (log)"); ax.set_ylabel("P(X ≥ x) (log) — over all comments of that kind")
ax.set_title("Tail of the distributions (CCDF; zeros sit below x = 1)")
ax.legend()

# (d) Lorenz
ax = axes[1, 1]
xs = np.arange(1, n_all + 1) / n_all
idx = np.unique(np.r_[np.arange(0, n_all, max(n_all // 2000, 1)), n_all - 1])
ax.plot(np.r_[0, xs[idx]], np.r_[0, cum_likes[idx]], color="#C44E52", lw=2, label=f"likes (Gini = {G:.2f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect equality")
ax.set_xlabel("cumulative share of comments (least → most liked)"); ax.set_ylabel("cumulative share of likes")
ax.set_title("Concentration of likes (Lorenz curve)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("Most-liked comments:")
display(df.nlargest(10, "like_count")[["like_count", "reply_count", "comment_type", "model", "text"]]
        .assign(text=lambda d: d["text"].str.slice(0, 110)))

### 3.3 Engagement by brand and model
* **Brand level:** the full distribution of like buckets, as 100 % stacked bars (top-level and replies separately, because replies attract fewer likes and brands differ in reply share).
* **Model level (≥ `MIN_N_MODEL` comments, top-level only):** the share of comments with ≥ 1 like, mean likes, mean replies received, and reply share, each with a **95 % interval** so that small models are not over-read.

The original per-model boxplots were removed: with > 75 % zeros every box collapses to a flat line and only outliers remain visible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8), sharey=True)
brand_order = brand_ct.index.tolist()
for ax, (ctype, d) in zip(axes, [("top-level", top), ("reply", rep)]):
    ct = (pd.crosstab(d["brand"], d["like_bucket"], normalize="index") * 100).reindex(brand_order).dropna(how="all")
    ct.plot.barh(stacked=True, ax=ax, color=sns.color_palette("rocket_r", n_colors=ct.shape[1]), width=0.8, legend=False)
    ax.set_xlabel("% of comments"); ax.set_title(f"Likes per comment by brand — {ctype}")
    ax.invert_yaxis()
axes[1].legend(title="likes", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
t = top.assign(liked_f=top["like_count"] > 0, replied=top["reply_count"] > 0)
ms = (t.groupby("model")
        .agg(n=("comment_id", "size"), p_liked=("liked_f", "mean"), mean_likes=("like_count", "mean"), sd_likes=("like_count", "std"),
             mean_replies=("reply_count", "mean"), sd_replies=("reply_count", "std")))
ms["reply_share"] = df.groupby("model")["is_reply"].mean()
ms["brand"] = ms.index.map(brand_map)
ms["se_p"] = np.sqrt(ms["p_liked"] * (1 - ms["p_liked"]) / ms["n"])
ms["se_likes"] = ms["sd_likes"] / np.sqrt(ms["n"])
ms["se_replies"] = ms["sd_replies"] / np.sqrt(ms["n"])

big = ms[ms["n"] >= MIN_N_MODEL].sort_values("n")
overall = {"p_liked": t["liked_f"].mean(), "mean_likes": t["like_count"].mean(),
           "mean_replies": t["reply_count"].mean(), "reply_share": df["is_reply"].mean()}

panels = [("p_liked", "se_p", "P(comment has ≥ 1 like)", True),
          ("mean_likes", "se_likes", "Mean likes per comment", False),
          ("mean_replies", "se_replies", "Mean replies received", False),
          ("reply_share", None, "Share of comments that are replies", True)]
fig, axes = plt.subplots(1, 4, figsize=(20, 0.3 * len(big) + 2.6), sharey=True)
for ax, (col, se, title, is_pct) in zip(axes, panels):
    scale = 100 if is_pct else 1
    err = None if se is None else 1.96 * big[se] * scale
    ax.barh(big.index, big[col] * scale, xerr=err, color=big["brand"].map(BRAND_COLORS), ecolor="black", capsize=2)
    ax.axvline(overall[col] * scale, color="red", ls="--", lw=1)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("%" if is_pct else "")
axes[0].set_yticks(range(len(big)))
axes[0].set_yticklabels([f"{m} (n={n:,})" for m, n in zip(big.index, big["n"])])
fig.suptitle("Engagement by model — top-level comments (red dashed = overall; bars = 95 % CI; colour = brand)", y=1.0, fontweight="bold")
plt.tight_layout()
plt.show()

# is the difference between brands statistically + practically meaningful?
brand_groups = [g["like_count"].to_numpy() for b, g in top.groupby("brand") if len(g) >= MIN_N_MODEL]
if len(brand_groups) >= 2:
    H, p = stats.kruskal(*brand_groups)
    k, n_k = len(brand_groups), sum(len(g) for g in brand_groups)
    eps2 = max((H - k + 1) / (n_k - k), 0)
    print(f"Kruskal–Wallis on like_count across {k} brands: H = {H:,.1f}, p = {p:.2g}, ε² = {eps2:.4f} "
          "(ε² close to 0 = brand explains almost none of the variation; with large n, tiny p-values can accompany tiny effects).")

### 3.4 Do liked comments also get replies?
Pearson correlation on zero-inflated heavy-tailed counts is dominated by a handful of outliers; we report **Spearman** and show the joint distribution with a hexbin on `log1p` axes (the original log-scale scatter silently dropped every comment with 0 likes or 0 replies — most of the data). Restricted to top-level comments, since only they have `reply_count`.

In [ ]:
tl = top.dropna(subset=["reply_count"]).copy()
rho, p = stats.spearmanr(tl["like_count"], tl["reply_count"])
print(f"Spearman ρ(likes, replies received) over {len(tl):,} top-level comments = {rho:.3f} (p = {p:.2g})")

tl["reply_bucket"] = pd.cut(tl["reply_count"], REPLY_BINS, labels=REPLY_LABELS)
brand_rho = {}
for b, g in tl.groupby("brand"):
    if len(g) >= MIN_N_MODEL:
        brand_rho[b] = stats.spearmanr(g["like_count"], g["reply_count"])[0]
brand_rho = pd.Series(brand_rho).sort_values()

fig, axes = plt.subplots(1, 3, figsize=(19, 5.2), gridspec_kw={"width_ratios": [1, 1.1, 0.8]})
ax = axes[0]
hb = ax.hexbin(np.log1p(tl["reply_count"]), np.log1p(tl["like_count"]), gridsize=35, bins="log", mincnt=1, cmap="viridis")
tick_v = [0, 1, 3, 10, 30, 100, 300, 1000]
xt = [v for v in tick_v if np.log1p(v) <= np.log1p(tl["reply_count"].max())]
yt = [v for v in tick_v if np.log1p(v) <= np.log1p(tl["like_count"].max())]
ax.set_xticks(np.log1p(xt)); ax.set_xticklabels(xt); ax.set_yticks(np.log1p(yt)); ax.set_yticklabels(yt)
ax.set_xlabel("replies received"); ax.set_ylabel("likes"); ax.set_title("Joint distribution (hexbin, log colour; zeros kept)")
fig.colorbar(hb, ax=ax, label="comments")

ax = axes[1]
ct = pd.crosstab(tl["like_bucket"], tl["reply_bucket"], normalize="index") * 100
sns.heatmap(ct, annot=True, fmt=".0f", cmap="YlGnBu", cbar_kws={"label": "% of row"}, ax=ax)
ax.set_xlabel("replies received"); ax.set_ylabel("likes received")
ax.set_title("P(reply bucket | like bucket), % per row")

ax = axes[2]
if len(brand_rho):
    ax.barh(brand_rho.index, brand_rho.values, color=[BRAND_COLORS[b] for b in brand_rho.index])
    ax.axvline(rho, color="red", ls="--", lw=1)
    ax.set_xlabel("Spearman ρ"); ax.set_title("ρ by brand (red = overall)")
    annotate_barh(ax, "{:.2f}")
plt.tight_layout()
plt.show()

### 3.5 Time
All hours / weekdays are in **local time** (`LOCAL_TZ`). The original notebook used UTC, which shifts every daily pattern by 7 hours.

Yearly and monthly counts mostly reflect **which videos were collected**, not how much people commented in general — so we first look at *collection coverage per model*, and only then interpret the trends.

In [ ]:
cov = (df.groupby("model").agg(first=("published_at", "min"), last=("published_at", "max"), n=("comment_id", "size"))
         .assign(brand=lambda d: d.index.map(brand_map)).sort_values("first"))
cov["first_n"] = cov["first"].dt.tz_localize(None)
cov["last_n"] = cov["last"].dt.tz_localize(None)

fig, ax = plt.subplots(figsize=(12, 0.27 * len(cov) + 1.8))
y = np.arange(len(cov))
ax.hlines(y, cov["first_n"], cov["last_n"], colors=cov["brand"].map(BRAND_COLORS), lw=4, alpha=0.85)
ax.scatter(cov["first_n"], y, s=18, color="black", zorder=3)
ax.set_yticks(y); ax.set_yticklabels(cov.index)
ax.invert_yaxis()
ax.set_title("Comment time span per model (first → last comment; colour = brand)")
ax.set_xlabel("published_at")
plt.tight_layout()
plt.show()

In [ ]:
monthly = df.dropna(subset=["month_start"]).groupby(["month_start", "brand"]).size().unstack(fill_value=0)
full_idx = pd.date_range(monthly.index.min(), monthly.index.max(), freq="MS")
monthly = monthly.reindex(full_idx, fill_value=0)
brand_cols = [b for b in brand_ct.index if b in monthly.columns]

yearly = df.dropna(subset=["year"]).groupby(["year", "brand"]).size().unstack(fill_value=0)[brand_cols]
yearly.index = yearly.index.astype(int)
moy = (pd.crosstab(df["brand"], df["month"], normalize="index") * 100).reindex(brand_cols)
moy.columns = [pd.Timestamp(2000, m, 1).strftime("%b") for m in moy.columns]

fig = plt.figure(figsize=(17, 11))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.05])
ax = fig.add_subplot(gs[0, :])
ax.stackplot(monthly.index, [monthly[b] for b in brand_cols], labels=brand_cols, colors=[BRAND_COLORS[b] for b in brand_cols], alpha=0.9)
ax.set_ylabel("comments per month"); ax.set_title("Monthly comment volume by brand")
ax.legend(loc="upper left", ncol=len(brand_cols))

ax = fig.add_subplot(gs[1, 0])
yearly.plot.bar(stacked=True, ax=ax, color=[BRAND_COLORS[b] for b in brand_cols], width=0.8, legend=False)
last_year = yearly.index.max()
ax.set_title("Comments per year by brand")
ax.set_xlabel("year (local)"); ax.set_ylabel("comments")
if df["published_at"].max().tz_convert(LOCAL_TZ).month < 12:
    ax.text(len(yearly) - 1, yearly.iloc[-1].sum(), " partial\n year", ha="center", va="bottom", fontsize=8)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{int(v):,}"))

ax = fig.add_subplot(gs[1, 1])
sns.heatmap(moy, annot=True, fmt=".0f", cmap="mako_r", cbar_kws={"label": "% of the brand's comments"}, ax=ax)
ax.set_title("Seasonality: share of each brand's comments by month")
ax.set_xlabel("month"); ax.set_ylabel("")
plt.tight_layout()
plt.show()

# test the 'launch-cycle' hypothesis instead of asserting it
sep_share = moy["Sep"] if "Sep" in moy.columns else pd.Series(dtype=float)
print("Share of each brand's comments posted in September (uniform would be 8.3 %):")
display(sep_share.round(1).to_frame("September %").T)
print("Caveat: monthly peaks can also come from *which* videos were crawled (see the coverage chart above), "
      "not only from launch events. Cross-check with video upload dates before drawing conclusions.")

In [ ]:
hourly_local = df["hour"].value_counts(normalize=True).sort_index() * 100
hourly_utc = df["published_at"].dt.hour.value_counts(normalize=True).sort_index() * 100
dow_lab = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow = df["dow"].value_counts(normalize=True).sort_index() * 100
heat = df.groupby(["dow", "hour"]).size().unstack(fill_value=0).reindex(index=range(7), columns=range(24), fill_value=0)
heat.index = dow_lab

fig = plt.figure(figsize=(16, 9))
gs = fig.add_gridspec(2, 2, width_ratios=[1.6, 1])
ax = fig.add_subplot(gs[0, 0])
ax.bar(hourly_local.index, hourly_local.values, color="#4C72B0", label=f"local ({LOCAL_TZ})")
ax.plot(hourly_utc.index, hourly_utc.values, color="#C44E52", marker="o", ms=3, lw=1.2, label="UTC (what the raw timestamps show)")
ax.set_xticks(range(0, 24, 2)); ax.set_xlabel("hour of day"); ax.set_ylabel("% of comments")
ax.set_title("Comments by hour of day"); ax.legend()
ax = fig.add_subplot(gs[0, 1])
ax.bar(dow_lab, dow.reindex(range(7)).values, color="#55A868")
ax.set_ylabel("% of comments"); ax.set_title("Comments by weekday (local)")
ax = fig.add_subplot(gs[1, :])
sns.heatmap(heat, cmap="YlGnBu", cbar_kws={"label": "comments"}, ax=ax, linewidths=0.2)
ax.set_xlabel("hour of day (local)"); ax.set_ylabel(""); ax.set_title("Weekday × hour")
plt.tight_layout()
plt.show()
print(f"Peak local hour: {int(hourly_local.idxmax()):02d}:00 | peak weekday: {dow_lab[int(dow.idxmax())]} | "
      f"(UTC peak would have been {int(hourly_utc.idxmax()):02d}:00)")

In [ ]:
# How quickly do replies arrive after the parent comment?
rl = rep[rep["parent_id"].isin(top_unique.index)]
lat_h = (rl["published_at"] - rl["parent_id"].map(top_unique["published_at"])).dt.total_seconds() / 3600
lat_h = lat_h[lat_h >= 0].dropna()
if len(lat_h):
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
    ax = axes[0]
    ax.hist(np.log10(lat_h.clip(lower=1 / 60)), bins=50, color="#4C72B0", edgecolor="white")
    marks = {"1 min": 1 / 60, "10 min": 1 / 6, "1 h": 1, "1 day": 24, "1 week": 168, "30 days": 720, "1 year": 8760}
    marks = {k: v for k, v in marks.items() if v <= lat_h.max() * 3}
    xlim = (np.log10(1 / 60) - 0.15, np.log10(max(lat_h.max(), 2)) + 0.2)
    ax.set_xticks(np.log10(list(marks.values()))); ax.set_xticklabels(list(marks.keys()), rotation=30)
    ax.axvline(np.log10(max(lat_h.median(), 1 / 60)), color="red", ls="--", label=f"median = {lat_h.median():.1f} h")
    ax.set_xlim(xlim)
    ax.set_xlabel("delay between parent comment and reply (log scale)"); ax.set_ylabel("replies")
    ax.set_title("Reply latency"); ax.legend()
    ax = axes[1]
    xs = np.sort(lat_h.to_numpy())
    ax.plot(np.log10(np.clip(xs, 1 / 60, None)), np.arange(1, len(xs) + 1) / len(xs), color="#DD8452")
    ax.set_xticks(np.log10(list(marks.values()))); ax.set_xticklabels(list(marks.keys()), rotation=30)
    ax.set_xlim(xlim)
    ax.set_ylabel("cumulative share of replies"); ax.set_title("Reply latency — ECDF")
    for h, lab in [(1, "1 h"), (24, "1 day")]:
        share = (lat_h <= h).mean()
        ax.axvline(np.log10(h), color="grey", ls=":"); ax.text(np.log10(h), 0.05, f" {share:.0%} within {lab}", fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# Life-cycle: how does discussion of a model decay after its first collected comment?
top8 = df["model"].value_counts().head(8).index
sub = df[df["model"].isin(top8)].copy()
sub["age_month"] = ((sub["published_at"] - sub.groupby("model")["published_at"].transform("min")).dt.days // 30).clip(upper=36)
life = sub.groupby(["age_month", "model"]).size().unstack(fill_value=0)
life = life.div(life.sum(), axis=1) * 100
fig, ax = plt.subplots(figsize=(12, 5))
pal = sns.color_palette("tab10", n_colors=len(top8))
for c, m in zip(pal, top8):
    ax.plot(life.index, life[m], marker="o", ms=3, lw=1.5, color=c, alpha=0.9, label=f"{m} ({brand_map[m]})")
ax.set_xlabel("months since the model's first collected comment (36 = 36+)")
ax.set_ylabel("% of the model's comments")
ax.set_title("Attention life-cycle of the 8 most-commented models")
ax.legend(ncol=4, fontsize=8)
plt.tight_layout()
plt.show()
print("Caveat: 'first collected comment' only approximates the release date; it depends on the oldest video that was crawled.")

### 3.6 Text characteristics
Length distributions (log axis — the raw histogram in the original is a single spike at the left), how length relates to engagement (binned rates with confidence intervals instead of an over-plotted scatter), differences by model, and how writing style changes over the years.

In [ ]:
q = [0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
len_tbl = pd.DataFrame({
    f"{c} — {ct}": d[c].quantile(q).rename(lambda x: f"p{x * 100:g}")
    for ct, d in [("top-level", top), ("reply", rep)] for c in ("text_len", "word_count")
}).T
display(len_tbl)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
for ax, col, xl in [(axes[0], "text_len", "characters"), (axes[1], "word_count", "whitespace-separated words")]:
    sns.histplot(data=df.assign(v=df[col].clip(lower=1)), x="v", hue="comment_type", log_scale=True, bins=50,
                 stat="density", common_norm=False, element="step", fill=True, alpha=0.25,
                 palette={"top-level": "#4C72B0", "reply": "#DD8452"}, ax=ax)
    ax.set_xlabel(f"comment length ({xl}, log scale)"); ax.set_title(f"Length distribution — {xl}")
plt.tight_layout()
plt.show()

In [ ]:
WC_BINS = [0, 1, 2, 3, 5, 8, 12, 20, 35, 60, np.inf]
WC_LABELS = ["1", "2", "3", "4–5", "6–8", "9–12", "13–20", "21–35", "36–60", "61+"]
d = df.assign(wc_bin=pd.cut(df["word_count"], WC_BINS, labels=WC_LABELS))
d["replied"] = d["reply_count"] > 0
rate = (d.groupby(["comment_type", "wc_bin"], observed=True)
          .agg(n=("comment_id", "size"), p_liked=("liked", "mean"), p_replied=("replied", "mean")).reset_index())
rate["se_liked"] = np.sqrt(rate["p_liked"] * (1 - rate["p_liked"]) / rate["n"])
rate["se_replied"] = np.sqrt(rate["p_replied"] * (1 - rate["p_replied"]) / rate["n"])

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for ct, c in [("top-level", "#4C72B0"), ("reply", "#DD8452")]:
    g = rate[rate["comment_type"] == ct]
    axes[0].errorbar(g["wc_bin"].astype(str), g["p_liked"] * 100, yerr=1.96 * g["se_liked"] * 100, marker="o", capsize=3, label=ct, color=c)
g = rate[rate["comment_type"] == "top-level"]
axes[1].errorbar(g["wc_bin"].astype(str), g["p_replied"] * 100, yerr=1.96 * g["se_replied"] * 100, marker="o", capsize=3, color="#55A868")
axes[0].set_title("Longer comments are liked more often?"); axes[0].set_ylabel("% with ≥ 1 like"); axes[0].legend()
axes[1].set_title("…and get more replies? (top-level)"); axes[1].set_ylabel("% with ≥ 1 reply")
for ax in axes:
    ax.set_xlabel("comment length (words)")
plt.tight_layout()
plt.show()
print("Comments per length bin:", rate.groupby("wc_bin", observed=True)["n"].sum().to_dict())

In [ ]:
lm = (df[df["model"].isin(ms[ms["n"] >= MIN_N_MODEL].index)]
      .groupby("model")["word_count"].describe(percentiles=[0.25, 0.5, 0.75])
      .sort_values("50%"))
fig, ax = plt.subplots(figsize=(9, 0.3 * len(lm) + 1.8))
colors = [BRAND_COLORS[brand_map[m]] for m in lm.index]
ax.barh(lm.index, lm["50%"], color=colors)
ax.errorbar(lm["50%"], np.arange(len(lm)), xerr=[lm["50%"] - lm["25%"], lm["75%"] - lm["50%"]], fmt="none", ecolor="black", capsize=2, lw=1)
ax.set_xlabel("words per comment — median with interquartile range")
ax.set_title("Comment length by model (models with ≥ MIN_N_MODEL comments)")
plt.tight_layout()
plt.show()

In [ ]:
yr = df.dropna(subset=["year"]).groupby("year").agg(
    n=("comment_id", "size"),
    median_words=("word_count", "median"),
    no_diacritics=("script_style", lambda s: (s == "Latin letters, no diacritics").mean() * 100),
    emoji=("has_emoji", lambda s: s.mean() * 100))
yr = yr[yr["n"] >= 500]
yr.index = yr.index.astype(int)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2), sharex=True)
for ax, (col, title, c) in zip(axes, [("median_words", "Median words per comment", "#4C72B0"),
                                       ("no_diacritics", "% comments in Latin letters without diacritics", "#DD8452"),
                                       ("emoji", "% comments containing emoji", "#55A868")]):
    ax.plot(yr.index, yr[col], marker="o", color=c)
    ax.set_title(title, fontsize=11); ax.set_xlabel("year (years with ≥ 500 comments)")
    ax.set_xticks(yr.index); ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### 3.7 Vocabulary and discussion topics
The original used `[a-zA-Z0-9]+` to tokenise, which **cuts every Vietnamese word at its first accented letter** (hence meaningless tokens such as `nhi`, `thi`, `ang`, `ngh` in its top-terms chart) and used an empty stopword list.
Here:
* Unicode-aware tokeniser on NFC-normalised, lower-cased text (`[^\W_]+`), digits and 1-character tokens removed (single letters are mostly chat shorthand).
* A starter Vietnamese stopword list (accented **and** accent-stripped spellings). *Extend it* after looking at the charts.
* Because Vietnamese words are often two syllables ("điện thoại", "pin trâu"), **bigrams** are shown next to unigrams.
* Brand-specific vocabulary uses a **log-odds ratio with smoothing** (z-score), which surfaces words that are *distinctive*, not merely frequent.
* Topic coverage uses a small **aspect lexicon** matched on accent-stripped text (so it also catches comments typed without diacritics). It is a heuristic — keywords such as `gia`, `dat`, `cam` are ambiguous.

In [ ]:
STOPWORDS = set("""
và là của có không ko khong thì mà được này đó đấy cho với một những các em anh chị ad mình tôi tui tớ nó rất cũng nhiều như khi
để đã sẽ vẫn còn nên hơn lại ra vào trên dưới rồi luôn thế vậy nhé nha nhỉ ạ à ơi ah ok gì sao ai bạn mọi người nếu vì bị đi đang
về nào nữa thôi cứ tại bởi mới đều hết qua lên xuống ở từ đến đây kia ấy thấy biết muốn phải làm xem rồi cái con thằng ông bác
chứ hả hử nhỉ nhá dạ vâng ừ ờ uk ừm hic haha hihi kkk kk the and for you
""".split())
STOPWORDS |= {strip_accents_str(w) for w in STOPWORDS}

TOKEN_RE = re.compile(r"[^\W_]+")
norm_text = txt.str.normalize("NFC").str.lower()
tokens = norm_text.map(TOKEN_RE.findall)


def keep(t: str) -> bool:
    return len(t) >= 2 and not t.isdigit() and t not in STOPWORDS


content = tokens.map(lambda ts: [t for t in ts if keep(t)])
bigram_lists = tokens.map(lambda ts: [f"{a} {b}" for a, b in zip(ts, ts[1:]) if keep(a) and keep(b)])

uni = Counter(chain.from_iterable(content))
bi = Counter(chain.from_iterable(bigram_lists))
n_tok = sum(uni.values())
print(f"{n_tok:,} content tokens | vocabulary {len(uni):,} | words seen once: {sum(1 for c in uni.values() if c == 1) / max(len(uni), 1):.1%} "
      f"| median {content.map(len).median():.0f} content tokens per comment")

fig, axes = plt.subplots(1, 3, figsize=(20, 7), gridspec_kw={"width_ratios": [1, 1, 0.9]})
for ax, counter, title, color in [(axes[0], uni, "Top unigrams", "#4C72B0"), (axes[1], bi, "Top bigrams", "#55A868")]:
    items = counter.most_common(25)[::-1]
    ax.barh([w for w, _ in items], [c for _, c in items], color=color)
    ax.set_title(f"{title} (stopwords removed)"); ax.set_xlabel("occurrences")
    annotate_barh(ax)
ax = axes[2]
freqs = np.array(sorted(uni.values(), reverse=True))
ax.loglog(np.arange(1, len(freqs) + 1), freqs, color="#C44E52")
ax.set_xlabel("word rank (log)"); ax.set_ylabel("frequency (log)"); ax.set_title("Zipf plot of the vocabulary")
plt.tight_layout()
plt.show()

In [ ]:
main_brands = [b for b in brand_ct.index if b not in ("Mixed", "Other") and brand_ct[b] >= MIN_N_MODEL][:4]
brand_counters = {b: Counter(chain.from_iterable(content[df["brand"] == b])) for b in main_brands}


def distinctive_terms(a: Counter, b: Counter, min_count=30, alpha=0.5, k=12):
    """Monroe et al. log-odds with uniform Dirichlet prior; returns (over-represented in a, over-represented in b)."""
    vocab = [w for w in set(a) | set(b) if a[w] + b[w] >= min_count]
    if not vocab:
        return pd.Series(dtype=float), pd.Series(dtype=float)
    na, nb, a0 = sum(a.values()), sum(b.values()), alpha * len(vocab)
    z = {}
    for w in vocab:
        ya, yb = a[w] + alpha, b[w] + alpha
        delta = np.log(ya / (na + a0 - ya)) - np.log(yb / (nb + a0 - yb))
        z[w] = delta / np.sqrt(1 / ya + 1 / yb)
    z = pd.Series(z).sort_values()
    return z.tail(k)[::-1], z.head(k)


if main_brands:
    fig, axes = plt.subplots(1, len(main_brands), figsize=(5.2 * len(main_brands), 5.6), squeeze=False)
    for ax, b in zip(axes[0], main_brands):
        rest = Counter()
        for ob in brand_ct.index:
            if ob != b:
                rest.update(Counter(chain.from_iterable(content[df["brand"] == ob])))
        over, _ = distinctive_terms(brand_counters[b], rest)
        ax.barh(over.index[::-1], over.values[::-1], color=BRAND_COLORS[b])
        ax.set_title(f"Distinctive of {b}"); ax.set_xlabel("log-odds z-score vs. all other brands")
    plt.tight_layout()
    plt.show()

In [ ]:
ASPECTS = {
    "Battery & charging":   r"\b(?:pin|sac|mah|sac nhanh)\b",
    "Camera & photo":       r"\b(?:camera|cam|chup|selfie|quay phim|quay video|zoom)\b",
    "Price & value":        r"\b(?:gia|tien|trieu|re|dat|giam gia|dang tien)\b",
    "Performance & gaming": r"\b(?:chip|snapdragon|bionic|muot|lag|giat|hieu nang|game|fps|ram|cau hinh)\b",
    "Display":              r"\b(?:man hinh|oled|amoled|hz|do sang|tan so quet)\b",
    "Heat":                 r"\b(?:nong|tan nhiet|nhiet)\b",
    "Design & build":       r"\b(?:dep|thiet ke|khung|vien|chat lieu)\b",
    "Software & updates":   r"\b(?:ios|android|update|cap nhat|one ui|hyperos|miui|phan mem)\b",
    "Warranty & retail":    r"\b(?:bao hanh|chinh hang|tgdd|the gioi di dong|fpt|cellphones|hoang ha|vn/a|quoc te|xach tay)\b",
}
aspect_df = pd.DataFrame({k: df["text_ascii"].str.contains(v, regex=True) for k, v in ASPECTS.items()})
overall_aspect = aspect_df.mean().mul(100).sort_values()
by_brand = aspect_df.groupby(df["brand"]).mean().mul(100).T.reindex(overall_aspect.index[::-1])
by_brand = by_brand[[b for b in brand_ct.index if b in by_brand.columns and brand_ct[b] >= MIN_N_MODEL]]

fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [0.8, 1.2]})
axes[0].barh(overall_aspect.index, overall_aspect.values, color="#4C72B0")
axes[0].set_xlabel("% of comments mentioning the aspect"); axes[0].set_title("What do people talk about?")
annotate_barh(axes[0], "{:.1f}%")
lift = by_brand.div(overall_aspect.reindex(by_brand.index), axis=0)
sns.heatmap(lift, annot=by_brand.to_numpy(), fmt=".1f", cmap="RdBu_r", center=1, vmin=0.5, vmax=1.5,
            cbar_kws={"label": "rate relative to all comments (1 = average)"}, ax=axes[1])
axes[1].set_title("Aspect mentions by brand (cell = % of comments, colour = over/under-indexing)")
axes[1].set_xlabel(""); axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Who is mentioned where? (video brand = row, brand named in the comment = column)
MENTIONS = {
    "Apple":   r"\b(?:iphone|ip ?\d+|apple|ios)\b",
    "Samsung": r"\b(?:samsung|sam sung|galaxy|ss|note ?\d+)\b",
    "Xiaomi":  r"\b(?:xiaomi|xiao mi|redmi|mi ?\d+|poco)\b",
    "Huawei":  r"\b(?:huawei|mate ?\d+)\b",
    "Oppo":    r"\b(?:oppo|reno ?\d*|find ?x\d*)\b",
    "vivo":    r"\b(?:vivo)\b",
}
men = pd.DataFrame({b: df["text_ascii"].str.contains(p, regex=True) for b, p in MENTIONS.items()})
rows = [b for b in MENTIONS if b in brand_ct.index and brand_ct[b] >= MIN_N_MODEL]
cross = men.groupby(df["brand"]).mean().mul(100).reindex(rows)
if len(cross):
    fig, ax = plt.subplots(figsize=(8, 0.7 * len(rows) + 2.4))
    sns.heatmap(cross, annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": "% of comments"}, ax=ax)
    ax.set_xlabel("brand mentioned in the comment"); ax.set_ylabel("brand of the video")
    ax.set_title("Cross-brand talk: off-diagonal = comparisons / switching")
    plt.tight_layout()
    plt.show()

### 3.8 Threads and authors

In [ ]:
obs = rep.groupby("parent_id").size()
tl2 = top.assign(observed_replies=top["comment_id"].map(obs).fillna(0).astype(int))
tb = pd.cut(tl2["observed_replies"], [-1, 0, 1, 2, 3, 5, 10, np.inf], labels=["0", "1", "2", "3", "4–5", "6–10", "11+"])
tb = tb.value_counts(normalize=True).sort_index() * 100

main = [b for b in brand_ct.index if b not in ("Mixed", "Other") and brand_ct[b] >= MIN_N_MODEL]
auth_brand = (pd.crosstab(df["author_hash"], df["brand"])[main] > 0) if main else pd.DataFrame()

fig, axes = plt.subplots(1, 3, figsize=(20, 5), gridspec_kw={"width_ratios": [0.9, 1.1, 0.9]})
ax = axes[0]
ax.bar(tb.index.astype(str), tb.values, color="#4C72B0")
for i, v in enumerate(tb.values):
    ax.text(i, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=8)
ax.set_xlabel("replies present in the file"); ax.set_ylabel("% of top-level comments"); ax.set_title("Thread size")

ax = axes[1]
n_models_per_author = df.groupby("author_hash")["model"].nunique()
vc = n_models_per_author.clip(upper=6).value_counts().sort_index()
ax.bar([str(i) if i < 6 else "6+" for i in vc.index], vc.values, color="#55A868")
ax.set_xlabel("distinct models an author commented on"); ax.set_ylabel("authors"); ax.set_title("Do authors follow several models?")

ax = axes[2]
if len(main) >= 2:
    co = auth_brand.T.astype(int).dot(auth_brand.astype(int))
    share = co.div(np.diag(co), axis=0) * 100
    sns.heatmap(share, annot=True, fmt=".0f", cmap="Purples", cbar_kws={"label": "% of row-brand authors"}, ax=ax)
    ax.set_title("Author overlap between brands")
    ax.set_xlabel("also commented on…"); ax.set_ylabel("authors who commented on…")
else:
    ax.set_axis_off()
plt.tight_layout()
plt.show()

multi_brand = (auth_brand.sum(axis=1) >= 2).mean() if len(main) >= 2 else np.nan
print(f"Authors active on ≥ 2 models: {pct((n_models_per_author >= 2).mean())} | on ≥ 2 brands: {pct(multi_brand) if multi_brand == multi_brand else 'n/a'}")
print("Largest threads:")
display(tl2.nlargest(6, "observed_replies")[["observed_replies", "reply_count", "like_count", "model", "text"]]
        .assign(text=lambda d: d["text"].str.slice(0, 100)))

## 4. Làm sạch và tách mệnh đề

Thứ tự ưu tiên: (1) nếu có `clean_data.jsonl` thì nạp; (2) nếu import được `run_cleaning` thì chạy pipeline của nhóm (đặt `RUN_PIPELINE = True`); (3) nếu không, dùng **bản dự phòng** bên dưới, chỉ xấp xỉ pipeline thật (cùng cờ `is_spam`, `is_low_info`, `mention_removed`, `clauses`) để các phần sau chạy được. Số liệu báo cáo nên lấy từ pipeline thật.

Luật spam thận trọng: chỉ gắn cờ khi có URL, số điện thoại hoặc lời mời liên hệ rõ ràng. Cờ không phải nhãn đã xác nhận; precision chưa đo.

In [ ]:
CLEAN_PATH = Path("./clean_data.jsonl")
TRACE_PATH = Path("./cleaning_trace.jsonl")
SUMMARY_PATH = Path("./cleaning_summary.json")
RUN_PIPELINE = False      # True: chạy lại run_cleaning.clean_dataset_with_trace (ghi đè 3 file đầu ra, không đụng file gốc)

URL_RE = re.compile(r"https?://\S+|www\.\S+", re.I)
PHONE_RE = re.compile(r"(?<!\d)0[35789](?:[ .\-]?\d){8}(?!\d)")
INVITE_RE = re.compile(r"(?:liên hệ|nhắn|inbox|ib|add|kết bạn)\s+(?:zalo|telegram|fb|facebook|sđt|sdt)", re.I)
MENTION_RE = re.compile(r"^\s*@\S+\s*")
SPLIT_RE = re.compile(r"[.!?;\n]+|,\s+|\s+(?:nhưng mà|nhưng|tuy nhiên|tuy)\s+", re.I)   # không tách ở "còn", "mà"
EMOJI_PUNCT_RE = re.compile(r"[\W_]+", re.U)


def split_clauses(text):
    parts = [p.strip() for p in SPLIT_RE.split(text) if p and p.strip()]
    return [p for p in parts if len(EMOJI_PUNCT_RE.sub("", p)) >= 2] or ([text.strip()] if text.strip() else [])


def fallback_clean(src: pd.DataFrame) -> pd.DataFrame:
    """Bản dự phòng: xấp xỉ pipeline thật. Không thay thế run_cleaning."""
    import html
    t = src["text"].fillna("").map(lambda s: unicodedata.normalize("NFC", html.unescape(s)))
    t = t.str.replace(r"\s+", " ", regex=True).str.strip()
    out = pd.DataFrame({"comment_id": src["comment_id"]})
    out["mention_removed"] = t.str.match(MENTION_RE)
    body = t.map(lambda s: MENTION_RE.sub("", s, count=1))
    out["text_norm"] = body.str.lower()
    reasons = pd.DataFrame({"contains_url": body.str.contains(URL_RE),
                            "contains_phone_number": body.str.contains(PHONE_RE),
                            "external_contact_invite": body.str.contains(INVITE_RE)})
    out["is_spam"] = reasons.any(axis=1)
    out["spam_reasons"] = reasons.apply(lambda r: [k for k, v in r.items() if v], axis=1)
    letters = body.map(lambda s: len(EMOJI_PUNCT_RE.sub("", s)))
    out["is_low_info"] = letters.lt(2) | body.map(lambda s: bool(EMOJI_RE.fullmatch(s.strip())) if s.strip() else True)
    cnt = src["author_hash"].map(src["author_hash"].value_counts())
    out["author_comment_count"] = cnt.to_numpy()
    out["is_channel_owner_candidate"] = out["author_comment_count"] > 50
    out["clauses"] = body.map(lambda s: [{"clause_id": i, "text": c, "text_norm": c.lower()}
                                         for i, c in enumerate(split_clauses(s))])
    out["clause_count"] = out["clauses"].map(len)
    return out


pipeline_available = False
try:
    from run_cleaning import clean_dataset_with_trace
    pipeline_available = True
except ImportError:
    pass

if RUN_PIPELINE and pipeline_available:
    clean_dataset_with_trace(input_path=str(DATA_PATH), output_path=str(CLEAN_PATH), trace_path=str(TRACE_PATH),
                             summary_path=str(SUMMARY_PATH), trace_only_changes=True, limit=0)

if CLEAN_PATH.exists():
    clean, bad = load_jsonl(CLEAN_PATH)
    CLEAN_SOURCE = "clean_data.jsonl"
else:
    clean = fallback_clean(df)
    CLEAN_SOURCE = "bản dự phòng trong notebook (xấp xỉ, không thay run_cleaning)"
print("Nguồn dữ liệu sạch:", CLEAN_SOURCE)

need = ["comment_id", "is_spam", "is_low_info", "mention_removed", "is_channel_owner_candidate", "clauses"]
miss = [c for c in need if c not in clean.columns]
assert not miss, f"Thiếu cột trong dữ liệu sạch: {miss}"
tn = clean[["comment_id", "text_norm"]].rename(columns={"text_norm": "text_clean"}) if "text_norm" in clean.columns else None
df = df.merge(clean[need], on="comment_id", how="left", validate="one_to_one")
if tn is not None:
    df = df.merge(tn, on="comment_id", how="left")
for c in ["is_spam", "is_low_info", "mention_removed", "is_channel_owner_candidate"]:
    df[c] = df[c].fillna(False).astype(bool)
df["clauses"] = df["clauses"].map(lambda v: v if isinstance(v, list) else [])
df["clause_count"] = df["clauses"].map(len)

n = len(df)
summary_clean = pd.DataFrame({
    "bản ghi": [int(v) for v in [df.is_spam.sum(), df.is_low_info.sum(), df.mention_removed.sum(),
                                  df.is_channel_owner_candidate.sum(), df.clause_count.sum()]],
}, index=["Gắn cờ spam", "Ít thông tin", "Bỏ mention đầu dòng", "Tác giả > 50 bình luận (ứng viên chủ kênh)", "Mệnh đề tách được"])
summary_clean["% bình luận"] = [df.is_spam.mean() * 100, df.is_low_info.mean() * 100, df.mention_removed.mean() * 100,
                                df.is_channel_owner_candidate.mean() * 100, np.nan]
display(summary_clean)
print(f"Trung bình {df.clause_count.mean():.2f} mệnh đề / bình luận")
reason_counts = Counter(chain.from_iterable(clean["spam_reasons"])) if "spam_reasons" in clean.columns else {}
print("Lý do spam (có thể chồng lấp):", dict(reason_counts))

### 4.1 Kiểm tra tách mệnh đề trên mẫu

Lỗi đã biết của bản cũ: "không còn là kẻ tạo ra xu thế" bị cắt tại "còn" thành `không` và `là kẻ tạo ra xu thế`. Bản dự phòng không tách ở "còn", "mà". Xem mẫu để quyết định có cần sửa luật của pipeline thật.

In [ ]:
multi = df[df.clause_count >= 3].sample(min(8, int((df.clause_count >= 3).sum())), random_state=SEED) if (df.clause_count >= 3).any() else df.head(0)
for _, r in multi.iterrows():
    print("•", plot_label(r["text"], 140))
    for c in r["clauses"]:
        print("     |", c["text"] if isinstance(c, dict) else c)
fig, ax = plt.subplots(figsize=(7, 3.4))
df["clause_count"].clip(upper=8).value_counts().sort_index().plot.bar(ax=ax, color="#4C72B0")
ax.set_xlabel("Số mệnh đề / bình luận (8 = 8 trở lên)"); ax.set_ylabel("Bình luận"); ax.set_title("Phân bố số mệnh đề")
plt.tight_layout(); plt.show()

## 5. Khía cạnh và cảm xúc theo mệnh đề (luật tham khảo)

Một danh mục thống nhất: **9 khía cạnh chuyên biệt + GENERAL**. Hiện có ba bộ (8 khía cạnh trong EDA gốc, 9 + GENERAL ở đây, 10 theo UIT-ViSFD); nhóm cần chốt một bộ. Không tự gộp khía cạnh hiếm theo ngưỡng 3%.

Quy tắc đọc kết quả:
- Phân tích trên mệnh đề của bình luận **không spam và không ít thông tin**.
- Độ phủ regex **không phải recall**. Một mệnh đề có thể khớp nhiều khía cạnh.
- Cảm xúc: có cả tín hiệu khen và chê → `mixed`; không có tín hiệu → `unknown` (không đổi thành `neutral`). Không dùng emoji hay luật mỉa mai chưa kiểm định để ép nhãn.
- Mã `model` là ngữ cảnh video, không phải sản phẩm được nhận xét.

In [ ]:
ASPECT_RULES = {
    'PERFORMANCE': r'\b(?:hiệu\s+năng|chip|snapdragon|exynos|dimensity|ram|lag|giật|fps|chơi\s+game|đa\s+nhiệm|tản\s+nhiệt|cấu\s+hình)\b',
    'BATTERY': r'\b(?:pin|sạc|onscreen|mah|chai\s+pin|sập\s+nguồn)\b',
    'DISPLAY': r'\b(?:màn\s+hình|tấm\s+nền|oled|amoled|sọc\s+màn|tần\s+số\s+quét|120\s*hz|60\s*hz|tai\s+thỏ)\b',
    'CAMERA': r'\b(?:cam|camera|chụp|quay\s+phim|ống\s+kính|zoom|selfie|xóa\s+phông|chống\s+rung)\b',
    'DESIGN': r'\b(?:thiết\s+kế|ngoại\s+hình|khung\s+viền|mặt\s+lưng|titan|cầm\s+nắm|mỏng|chống\s+nước|ip68)\b',
    'SOFTWARE': r'\b(?:ios|android|one\s*ui|hyperos|miui|cập\s+nhật|phần\s+mềm|hệ\s+điều\s+hành|giao\s+diện)\b',
    'HARDWARE_OTHER': r'\b(?:loa|âm\s+thanh|wifi|4g|5g|vân\s+tay|face\s*id|type\s*c|nfc|esim|micro|tai\s+nghe)\b',
    'PRICE': r'\b(?:giá|đắt|rẻ|triệu|đáng\s+tiền|túi\s+tiền|phân\s+khúc|giảm\s+giá)\b',
    'AFTER_SALES': r'\b(?:bảo\s+hành|sửa\s+chữa|đổi\s+trả|linh\s+kiện|hậu\s+mãi|chăm\s+sóc\s+khách\s+hàng)\b',
    'GENERAL': r'\b(?:ngon|ổn|tuyệt\s+vời|quá\s+tốt|chán|tệ|đáng\s+mua)\b',
}
SPECIFIC = [a for a in ASPECT_RULES if a != 'GENERAL']
ASPECT_LABEL = {'PERFORMANCE': 'Hiệu năng', 'BATTERY': 'Pin & sạc', 'DISPLAY': 'Màn hình', 'CAMERA': 'Camera',
                'DESIGN': 'Thiết kế', 'SOFTWARE': 'Phần mềm', 'HARDWARE_OTHER': 'Phần cứng khác',
                'PRICE': 'Giá bán', 'AFTER_SALES': 'Hậu mãi', 'GENERAL': 'Đánh giá chung'}

POS_TERMS = ['không xấu', 'không tệ', 'không lag', 'không giật', 'không nóng', 'không lỗi',
             'đẹp', 'tốt', 'ngon', 'mượt', 'tuyệt', 'ổn', 'bền', 'thích', 'ưng', 'đáng tiền', 'hài lòng', 'ấn tượng', 'sắc nét', 'rẻ']
NEG_TERMS = ['không tốt', 'không đẹp', 'không ngon', 'không mượt', 'không thích', 'không ổn', 'không đáng', 'chưa tốt', 'chưa đẹp',
             'không hài lòng', 'xấu', 'tệ', 'kém', 'lag', 'giật', 'nóng', 'lỗi', 'thất vọng', 'chán', 'đắt', 'yếu', 'chậm', 'hỏng',
             'hao pin', 'tụt pin', 'chai pin']
term_sign = {**{t: 1 for t in POS_TERMS}, **{t: -1 for t in NEG_TERMS}}
SENT_RE = re.compile(r'(?<!\w)(?:' + '|'.join(re.escape(t) for t in sorted(term_sign, key=lambda s: -len(s))) + r')(?!\w)', re.I)
SENT_ORDER = ['positive', 'negative', 'mixed', 'unknown']


def weak_polarity(text):
    hits = [m.group().lower() for m in SENT_RE.finditer(text)]   # cụm dài được ưu tiên: "không tốt" không tính thêm "tốt"
    pos = any(term_sign[t] > 0 for t in hits)
    neg = any(term_sign[t] < 0 for t in hits)
    return 'mixed' if pos and neg else 'positive' if pos else 'negative' if neg else 'unknown'


def combine_labels(values):
    s = set(values) - {'unknown'}
    if not s:
        return 'unknown'
    if 'mixed' in s or {'positive', 'negative'} <= s:
        return 'mixed'
    return next(iter(s))


eligible = df.loc[~df.is_spam & ~df.is_low_info & df.clause_count.gt(0)]
rows = []
for cid, brand, model, ts, cls in zip(eligible.comment_id, eligible.brand, eligible.model, eligible.published_at, eligible.clauses):
    for k, c in enumerate(cls):
        txt = c["text"] if isinstance(c, dict) else str(c)
        rows.append((cid, k, txt, txt.lower(), brand, model, ts))
clauses = pd.DataFrame(rows, columns=["comment_id", "clause_id", "clause_text", "clause_lower", "brand", "model", "published_at"])
clauses["sentiment"] = clauses["clause_lower"].map(weak_polarity)
masks = pd.DataFrame({a: clauses.clause_lower.str.contains(p, regex=True, na=False) for a, p in ASPECT_RULES.items()})
print(f"{len(eligible):,} bình luận đủ điều kiện → {len(clauses):,} mệnh đề")

# độ phủ (không phải recall)
cov = (masks.mean() * 100).rename(index=ASPECT_LABEL).sort_values()
any_specific = masks[SPECIFIC].any(axis=1).mean() * 100
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
axes[0].barh(cov.index, cov.values, color="#4C72B0"); axes[0].axvline(3, color="#E08E0B", ls="--", lw=1, label="mốc tham khảo 3%")
axes[0].set_xlabel("% mệnh đề khớp luật"); axes[0].set_title("Độ phủ khía cạnh (regex, không phải recall)"); axes[0].legend(frameon=False)
annotate_barh(axes[0], "{:.1f}%")
sent_ct = clauses["sentiment"].value_counts().reindex(SENT_ORDER, fill_value=0)
axes[1].bar(sent_ct.index, sent_ct.values, color=["#2E8B57", "#C8102E", "#E08E0B", "#9E9E9E"])
axes[1].set_title("Cảm xúc mệnh đề theo luật từ khóa (tham khảo)")
for i, v in enumerate(sent_ct.values):
    axes[1].text(i, v, f"{v:,}\n({v / len(clauses) * 100:.1f}%)", ha="center", va="bottom", fontsize=8)
axes[1].set_ylim(0, sent_ct.max() * 1.2)
plt.tight_layout(); plt.show()
print(f"{any_specific:.1f}% mệnh đề khớp ít nhất một trong 9 khía cạnh chuyên biệt; phần còn lại là GENERAL hoặc không khớp.")

# bảng comment–khía cạnh (dùng cho phần so sánh và nhận mô hình thật về sau)
asp_rows = []
for a in ASPECT_RULES:
    sub = clauses.loc[masks[a], ["comment_id", "clause_text", "sentiment", "brand", "model"]].copy()
    sub["aspect"] = a
    asp_rows.append(sub)
clause_aspects = pd.concat(asp_rows, ignore_index=True)
comment_aspects = clause_aspects.groupby(["comment_id", "aspect"], as_index=False).agg(
    sentiment=("sentiment", combine_labels), model=("model", "first"), brand=("brand", "first"),
    evidence=("clause_text", lambda s: " | ".join(dict.fromkeys(s))[:500]))
print(f"{len(comment_aspects):,} cặp bình luận–khía cạnh")

### 5.1 Khía cạnh theo hãng và đối chiếu hai mẫu máy

Chỉ vẽ ô khi số mẫu đạt `MIN_SUPPORT`; ô vắng là thiếu mẫu, không phải 0% tiêu cực. Mẫu số gồm cả `unknown` và `mixed`. Mã `model` là ngữ cảnh video, nên đây chưa phải xếp hạng chất lượng sản phẩm.

In [ ]:
MIN_SUPPORT = 30
COMPARE_MODELS = ("ip17prm", "ip18prm")     # đổi theo cặp demo; nếu không có sẽ lấy 2 model nhiều bình luận nhất

neg_by_brand = (comment_aspects[comment_aspects.aspect.isin(SPECIFIC)]
                .assign(neg=lambda d: d.sentiment.eq("negative"))
                .groupby(["brand", "aspect"]).agg(n=("neg", "size"), neg=("neg", "mean")).reset_index())
neg_by_brand["neg"] = (neg_by_brand["neg"] * 100).where(neg_by_brand["n"] >= MIN_SUPPORT)
hm = neg_by_brand.pivot(index="brand", columns="aspect", values="neg").reindex(columns=SPECIFIC).rename(columns=ASPECT_LABEL)
fig, ax = plt.subplots(figsize=(11, 3.6))
sns.heatmap(hm, annot=True, fmt=".0f", cmap="Reds", vmin=0, linewidths=1, linecolor="white", ax=ax,
            cbar_kws={"label": f"% tiêu cực (n ≥ {MIN_SUPPORT})"})
ax.set_title("Tỷ lệ tiêu cực theo khía cạnh và hãng (luật tham khảo)"); ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); plt.show()

cm = [m for m in COMPARE_MODELS if m in set(comment_aspects.model)]
if len(cm) < 2:
    cm = comment_aspects.model.value_counts().index[:2].tolist()
    print("COMPARE_MODELS không có trong dữ liệu; dùng 2 model nhiều nhất:", cm)
pair = comment_aspects[comment_aspects.model.isin(cm) & comment_aspects.aspect.isin(SPECIFIC)]
tab = (pair.assign(neg=lambda d: d.sentiment.eq("negative"))
       .groupby(["model", "aspect"]).agg(n=("neg", "size"), neg=("neg", "mean")).reset_index())
tab["neg"] = (tab["neg"] * 100).where(tab["n"] >= MIN_SUPPORT)
fig, ax = plt.subplots(figsize=(9, 4.4))
yy = np.arange(len(SPECIFIC))
for m, col, off, mk in zip(cm, ["#2A788E", "#E08E0B"], [-0.13, 0.13], ["o", "D"]):
    t = tab[tab.model == m].set_index("aspect").reindex(SPECIFIC)
    ax.scatter(t["neg"], yy + off, s=55, color=col, marker=mk, label=m)
    for y, nn, r in zip(yy + off, t["n"], t["neg"]):
        if pd.notna(r):
            ax.annotate(f"n={int(nn)}", (r, y), xytext=(7, 0), textcoords="offset points", va="center", fontsize=8, color=col)
ax.set_yticks(yy, [ASPECT_LABEL[a] for a in SPECIFIC]); ax.set_xlim(0, 100); ax.grid(axis="x", alpha=.5); ax.legend(frameon=False)
ax.set_xlabel(f"% tiêu cực / cặp bình luận–khía cạnh (chỉ hiện khi n ≥ {MIN_SUPPORT})")
ax.set_title("Đối chiếu hai ngữ cảnh video (chưa phải đối chiếu sản phẩm)")
plt.tight_layout(); plt.show()

## 6. Replay cảnh báo từ timestamp thật

Đây là **replay lịch sử**, không phải hệ thống realtime. Không dùng dữ liệu ngẫu nhiên, không chèn đỉnh.

Quy ước tại mốc `t` (giờ địa phương): bin 10 phút đóng phải, cộng 6 bin thành cửa sổ `(t − 1h, t]`. Tỷ lệ tiêu cực = số bình luận `negative` / số bình luận có nhãn rõ (`positive`, `negative`, `mixed`). Ngưỡng = trung bình các mốc 24h trước + 2 độ lệch chuẩn (`shift(1)`, chỉ dùng quá khứ). Chỉ xét khi có ≥ 30 bình luận nhãn rõ, độ bao phủ ≥ 20%, ≥ 36 mốc quá khứ hợp lệ. Các tham số là mức demo, chưa tối ưu bằng nhãn sự kiện thật. Cửa sổ chồng lấp nên điểm cảnh báo không độc lập; cần gom thành đợt.

Kế hoạch triển khai muốn cảnh báo so với **đời trước ở cùng số ngày sau ra mắt**, vì comment của mỗi mẫu máy dồn vào khoảng ngắn nên 24h trước thường thiếu dữ liệu. Phần này mới làm bản 24h; so với đời trước là việc tiếp theo.

In [ ]:
import matplotlib.dates as mdates

REPLAY_MODEL = None          # ví dụ "ip18prm"; None = model có ngày đông bình luận nhất
KNOWN = ['positive', 'negative', 'mixed']

cs = clauses.groupby("comment_id")["sentiment"].agg(combine_labels)
ev = df.loc[~df.is_spam & ~df.is_low_info, ["comment_id", "model", "published_at"]].copy()
ev["sentiment"] = ev["comment_id"].map(cs).fillna("unknown")
ev["dt_local"] = ev["published_at"].dt.tz_convert(LOCAL_TZ)
if REPLAY_MODEL:
    ev = ev[ev.model == REPLAY_MODEL]
ev = ev.dropna(subset=["dt_local"])

day_counts = ev.groupby(ev.dt_local.dt.floor("D")).size()
alert_episodes = pd.DataFrame()
if len(day_counts) == 0:
    print("Không có dữ liệu để replay.")
else:
    peak_day = day_counts.idxmax()
    plot_start, plot_end = peak_day - pd.Timedelta(days=1), peak_day + pd.Timedelta(days=2)
    history_start = plot_start - pd.Timedelta(hours=25)
    e = ev[ev.dt_local.between(history_start, plot_end, inclusive="right")].copy()
    e["total"] = 1
    e["known"] = e.sentiment.isin(KNOWN).astype(int)
    e["negative"] = e.sentiment.eq("negative").astype(int)
    bins = e.set_index("dt_local").sort_index()[["total", "known", "negative"]].resample("10min", closed="right", label="right").sum()
    grid = pd.date_range(history_start.floor("10min"), plot_end.ceil("10min"), freq="10min")
    bins = bins.reindex(grid, fill_value=0)
    roll = bins.rolling(6, min_periods=6).sum()
    roll["negative_rate"] = roll.negative.div(roll.known.replace(0, np.nan)) * 100
    roll["coverage"] = roll.known.div(roll.total.replace(0, np.nan)) * 100
    ok = roll.known.ge(30) & roll.coverage.ge(20)
    past = roll.negative_rate.where(ok).shift(1)
    roll["threshold"] = past.rolling("24h", min_periods=36).mean() + 2 * past.rolling("24h", min_periods=36).std()
    roll["alert"] = ok & roll.threshold.notna() & roll.negative_rate.gt(roll.threshold)
    pl = roll.loc[(roll.index >= plot_start) & (roll.index <= plot_end)]
    al = pl[pl.alert]
    fig, axes = plt.subplots(3, 1, figsize=(12, 8.5), sharex=True)
    axes[0].plot(pl.index, pl.negative_rate, color="#2A788E", lw=1.6, label="% tiêu cực / nhãn rõ")
    axes[0].plot(pl.index, pl.threshold, color="#E4572E", ls="--", lw=1.3, label="ngưỡng từ 24h trước")
    axes[0].scatter(al.index, al.negative_rate, color="#E4572E", s=22, zorder=5, label=f"{len(al)} mốc cảnh báo")
    axes[0].set_ylim(0, 105); axes[0].set_ylabel("%"); axes[0].legend(frameon=False, ncol=3, fontsize=9)
    axes[1].plot(pl.index, pl.coverage, color="#3A6EA5"); axes[1].axhline(20, color="#E08E0B", ls=":"); axes[1].set_ylabel("% có nhãn rõ"); axes[1].set_ylim(0, 105)
    axes[2].fill_between(pl.index, pl.total, color="#DAEAE0", label="tổng bình luận / 1h")
    axes[2].plot(pl.index, pl.known, color="#2A788E", label="có nhãn rõ"); axes[2].axhline(30, color="#E08E0B", ls=":")
    axes[2].set_ylabel("bình luận / cửa sổ"); axes[2].legend(frameon=False, ncol=2, fontsize=9)
    axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%d/%m\n%H:%M"))
    fig.suptitle(f"Replay cảnh báo · ngày nhiều bình luận nhất {peak_day:%d/%m/%Y} · {LOCAL_TZ}", fontweight="bold")
    plt.tight_layout(); plt.show()
    if len(al):
        grp = al.index.to_series().diff().gt(pd.Timedelta(minutes=10)).cumsum()
        alert_episodes = (al.assign(t=al.index).groupby(grp.to_numpy())
                          .agg(start=("t", "min"), end=("t", "max"), peak_negative_rate=("negative_rate", "max"),
                               max_known=("known", "max"), alert_points=("alert", "sum")))
        display(alert_episodes)
    else:
        print("Không có đợt cảnh báo thỏa điều kiện trong đoạn replay (không chèn điểm giả để tạo hình).")

### 6.1 Xuất kết quả trung gian

Các bảng này là đầu vào cho nhóm gán nhãn và nhóm mô hình. Chúng là **nhãn yếu** từ luật, không phải nhãn vàng.

In [ ]:
OUT_DIR = Path("./eda_outputs"); OUT_DIR.mkdir(exist_ok=True)
clauses.drop(columns=["clause_lower"]).to_csv(OUT_DIR / "clauses_weak_labels.csv", index=False, encoding="utf-8-sig")
comment_aspects.to_csv(OUT_DIR / "comment_aspects_weak.csv", index=False, encoding="utf-8-sig")
if len(alert_episodes):
    alert_episodes.to_csv(OUT_DIR / "alert_episodes.csv", encoding="utf-8-sig")
pd.DataFrame(QA_LOG).to_csv(OUT_DIR / "data_quality_scorecard.csv", index=False, encoding="utf-8-sig")
print("Đã ghi:", sorted(p.name for p in OUT_DIR.iterdir()))

## 7. Summary


In [ ]:
n = len(df)
top_models = df["model"].value_counts()
top_brand = brand_ct.index[0]
peak_month = df["month"].value_counts().idxmax()
peak_month_name = pd.Timestamp(2000, int(peak_month), 1).strftime("%B")
peak_month_share = (df["month"] == peak_month).mean()
peak_year = int(df["year"].value_counts().idxmax())
apple_sep = float(moy.loc["Apple", "Sep"]) if "Apple" in moy.index and "Sep" in moy.columns else float("nan")
spec_style = df["script_style"].value_counts(normalize=True)
sub_dup = float((df["dup_group_n"].gt(1) & ~df["is_trivial"]).mean())
_sc = pd.DataFrame(QA_LOG)["status"].value_counts()
status_txt = " · ".join(f"{int(_sc.get(k, 0))} {k}" for k in ["FAIL", "WARN", "INFO", "PASS"])

md = f"""
### Dataset
* **{n:,}** comments from **{df['video_id'].nunique():,}** videos about **{df['model'].nunique()}** model labels ({df['brand'].nunique()} brand groups), `{df['published_at'].min():%Y-%m-%d}` → `{df['published_at'].max():%Y-%m-%d}`; source: {', '.join(df['source'].dropna().unique())}.
* **{pct(1 - df['is_reply'].mean())}** top-level comments, **{pct(df['is_reply'].mean())}** replies. `reply_count` exists only for top-level comments (kept as `NaN` for replies).
* Brand mix: **{top_brand}** is the largest group ({pct(brand_ct.iloc[0] / n)} of comments); the most-commented model is **{top_models.index[0]}** ({top_models.iloc[0]:,}), followed by {top_models.index[1]} and {top_models.index[2]}.

### Data quality ({status_txt})
* Referential integrity: {pct(rates.iloc[0] / 100)} of replies have their parent in the file; thread completeness — {pct(full_capture_share)} of top-level comments that have replies are fully captured; overall the file holds ≈ {len(rep) / max(total_rep, 1):.0%} of the replies YouTube reports.
* Substantive duplicate texts (> 4 chars): {pct(sub_dup, 2)} of rows; trivial repeats ("hay", "ok", emoji…) are expected and harmless.
* `text_norm` is empty for {pct(df['text_norm_blank'].mean())} of rows.
* Composite model labels (e.g. `ip8|note8`): {pct(df['model'].str.contains('|', regex=False).mean())} of rows.
* Writing style: {spec_style.get('Vietnamese with diacritics', 0) * 100:.1f}% with Vietnamese diacritics, {spec_style.get('Latin letters, no diacritics', 0) * 100:.1f}% Latin letters without diacritics, {spec_style.get('No letters (emoji / punctuation / digits)', 0) * 100:.1f}% emoji/punctuation only.

### Engagement
* Likes are extremely skewed: {pct((top['like_count'] == 0).mean())} of top-level comments and {pct((rep['like_count'] == 0).mean())} of replies have 0 likes; the top 1 % of comments hold {pct(share_top1)} of all likes (Gini {G:.2f}); maximum {int(df['like_count'].max()):,} likes.
* Among top-level comments, {pct((top['reply_count'] == 0).mean())} receive no reply; the maximum is {int(top['reply_count'].max()):,}.
* Likes and replies are {'positively but only moderately' if 0.2 < rho < 0.6 else 'strongly' if rho >= 0.6 else 'weakly'} related (Spearman ρ = {rho:.2f}).

### Time (local time, {LOCAL_TZ})
* Busiest year: **{peak_year}**; busiest calendar month: **{peak_month_name}** ({pct(peak_month_share)} of all comments). Apple's September share is {apple_sep:.1f} % (uniform = 8.3 %) — consistent with a launch-cycle effect, but confounded by which videos were collected.
* Peak hour: **{int(hourly_local.idxmax()):02d}:00** local (it would read {int(hourly_utc.idxmax()):02d}:00 in raw UTC); peak weekday: **{dow_lab[int(dow.idxmax())]}**.
* Median reply arrives after {lat_h.median():.1f} h; {pct((lat_h <= 24).mean()) if len(lat_h) else 'n/a'} within a day.

### Text
* Median length: {top['word_count'].median():.0f} words (top-level) vs {rep['word_count'].median():.0f} (replies); {pct((df['word_count'] <= 3).mean())} of comments have ≤ 3 words.
* Frequent topics (lexicon heuristic): {', '.join(overall_aspect.index[::-1][:3])}.
"""
display(Markdown(md))

### Recommended preprocessing before modelling
1. **Unicode**: `unicodedata.normalize("NFC", text)`; `html.unescape`; strip URLs / @mentions / timestamps if they are noise for the task.
2. **Diacritics**: add a diacritic-insensitive view of the text (`text_ascii`) or a restoration model — a large share of comments has none.
3. **Duplicates**: drop `dup_ingest`; de-duplicate on `text_key` (non-trivial texts) *before* any train/test split; consider filtering accounts flagged in §2.8.
4. **`reply_count`**: keep `NaN` for replies; never impute 0. Use `observed_replies` only for threads that are fully captured.
5. **Labels**: decide how to treat composite `model` labels (drop, split, or a "multi-model" class) and use `brand` for coarse analyses.
6. **`text_norm`**: re-generate or drop it if it is empty for a meaningful share of rows.
7. **Splitting**: split by `video_id` (or at least by time) rather than randomly — comments from one video share topic and vocabulary.
8. **Constant columns** (`source`, `schema_version`, and any constant timestamp) can be dropped.

### Limits of this analysis
* Time trends reflect the crawl design (which videos were collected) as much as audience behaviour.
* Aspect and brand-mention matching are keyword heuristics, not classifiers; no sentiment is estimated here.
* Engagement counts are a snapshot at `fetched_at`; recent comments have had less time to accumulate likes.

### Appendix — what changed relative to the first version of this notebook
| Area | Problem in v1 | Fix in this version |
|---|---|---|
| Execution | Cells depended on hidden state (`KeyError: 'day_of_week'`); one cell never ran | Linear, self-contained flow; derived columns in one place |
| Loading | `try/except: print(e)` left `df` undefined | Explicit loader, bad-line report, schema + dtype checks |
| `reply_count` | `fillna(0)` mixed "n/a" with "no replies" and distorted all stats | Kept `NaN`; reply stats on top-level comments only |
| Duplicates | Counted any repeated text; label hack applied to counts; emoji rendered as boxes | Trivial vs substantive split, accent/case-insensitive key, safe labels |
| Likes vs replies | Pearson on skewed data; log axes dropped zeros; unreadable 10-colour scatter | Spearman, hexbin on `log1p`, bucket heat-map, ρ by brand |
| Model comparison | Boxplots collapsed by zeros; mean-likes table ranked tiny models | Rates with 95 % CI, minimum-n filter, brand colours |
| Time | UTC hours; tables only, no plots; launch-cycle claim untested | Local time, coverage chart, stacked series, seasonality heat-map, tested claim |
| Tokenising | ASCII regex destroyed Vietnamese words; no stopwords | Unicode tokeniser, stopwords, bigrams, log-odds, aspect lexicon |
| Summary | Hard-coded numbers (and a garbled apostrophe) | Generated from the data |

### Nguồn của từng phần code và những gì đã bỏ

- **Giữ:** toàn bộ phần nạp, kiểm tra chất lượng và EDA của `ds_inspect_comments_v2` (loader báo dòng lỗi, `reply_count` giữ null, giờ GMT+7, tách trùng nội dung thường/đáng kể, khoảng tin cậy theo model).
- **Thay thế:** `eda_smartphone_comments` (EDA gốc) và `EDA_Smartphone_Comments_Visual_Pro` (+ bản `_1` giống hệt) về phần EDA; các biểu đồ top model, nhắc tên chéo, từ vựng đã có trong phần 3.
- **Bổ sung từ Visual Pro:** bộ khía cạnh 9 + GENERAL, luật cảm xúc theo mệnh đề (cụm dài ưu tiên, `mixed`/`unknown`), replay cảnh báo cửa sổ 1h bước 10 phút.
- **Bỏ:** mục 10, 11 của EDA gốc (chuỗi cảnh báo sinh bằng `np.random`, F1/precision gõ tay); loại spam bằng luật rộng; `fillna(0)` cho `reply_count`; ngưỡng 3% tự gộp khía cạnh.